# DeltaFlow: CFM + Delta-Alignment Pretraining (PoC)

**Research idea.** Reformulate CDPM-Align's multi-scale *guidance-alignment* pretraining by swapping its DDPM backbone for a **flow-matching (rectified-flow) velocity field**. Same anatomy-cancelling `Δh = h_cond − h_uncond` alignment mechanism, but the generative engine is continuous-time Conditional Flow Matching (CFM) instead of discrete DDPM noise prediction. The multi-scale guidance-alignment loss is a configurable component (`EXPERIMENT["align"]`); the current run **disables** it (`align=False`) as a *pure-flow* ablation that isolates the flow-matching backbone before alignment is added back.

This notebook is a **fully self-contained, one-click** proof-of-concept you can run on a free cloud GPU. It:
1. clones the [`deltaflow`](https://github.com/phrugsa-limbunlom/deltaflow) library and injects the conditional UNet backbone + training module,
2. **pretrains** the CFM velocity field on a **pooled Shenzhen + ISBI2015** corpus with **dataset-index class conditioning** — no manual annotations needed, and **every downstream test image is held out to avoid leakage** — and plots the EMA-smoothed flow / alignment loss curves,
3. runs a **downstream few-shot landmark-detection probe** (MRE / SDR / P95) comparing the *pretrained* backbone against a *random-init* baseline to measure transfer, and
4. upgrades the probe to **real anatomical lung landmarks** (ngaggion) so the evaluation reflects genuine anatomy rather than pseudo-labels.

---
### Free GPU options (no local GPU needed)
| Platform | GPU | Free quota | Notes |
|---|---|---|---|
| **Kaggle** (recommended) | T4 ×2 or P100 (16 GB) | ~30 h/week | Dataset lives on-platform → zero download. *Settings → Accelerator → GPU*. |
| Google Colab | T4 (16 GB) | a few h/session | Good fallback; mount data via `kagglehub`. |
| Lightning AI Studio | T4 | ~15 h/month free | Persistent env. |

> **On Kaggle:** *Settings → Accelerator → GPU T4 ×2 (or P100)*, then *Run All*. Everything below is CPU-safe too (just slower) for a quick smoke test.

## Experiment design — variables & controls

**Hypothesis (H1).** Replacing CDPM-Align's DDPM noise-prediction backbone with a
continuous-time **flow-matching** velocity field, while keeping the *same*
multi-scale guidance-alignment pretraining, yields **equal-or-better** few-shot
landmark metrics than CDPM-Align under an *identical* evaluation protocol.

The single `EXPERIMENT` dict in the next cell is the **one source of truth** for
every downstream cell, so the **controlled** variables are guaranteed identical
across the pretraining and all probes. **Change exactly one knob per run.**

> **Alignment switch.** `EXPERIMENT["align"]` turns the multi-scale guidance-alignment loss on/off. The current config uses `align=False` (a pure flow-matching ablation); set it to `True` for the faithful CDPM-Align method. Treat `align` as its own ablation axis.

| role | variable | this run |
|---|---|---|
| **Independent** (what we change) | generative backbone / objective | `cfm` (flow matching) vs `ddpm` (paper) |
| **Dependent** (the answer we read) | MRE ↓, SDR@2mm ↑, SDR@4mm ↑, P95 ↓ | measured by the probe |
| **Controlled** (held identical) | dataset, split, image_size, n_shot, feature levels S, λ_align, optimiser/lr, iterations, eval unit (`mm_per_pixel`), seed | see `EXPERIMENT` |

**Reference to beat** — CDPM-Align, ISBI2015, 25-shot: **MRE 1.54 mm, SDR@2mm 77.52%**
(compare in *millimetres* on the same dataset/split — never pixel-vs-mm).


In [ ]:
import os, sys, json, csv, random, datetime
from types import SimpleNamespace

# ============================================================================
# EXPERIMENT — single source of truth for the controlled + independent
# variables. Every downstream cell reads from here, so the CONTROLLED variables
# stay identical across compared runs and only the INDEPENDENT variable moves.
# ============================================================================
EXPERIMENT = {
    # --- INDEPENDENT variable (the ONE thing we vary to test H1) ------------
    "backbone": "ddpm",             # "cfm" (this work) | "ddpm" (paper repro) <- DDPM head-to-head baseline

    # --- CONTROLLED variables (MUST match across compared runs) -------------
    "dataset": "combined",          # CDPM-Align pooled corpus (Shenzhen+ISBI2015+DHA)
    "split": "train",
    "image_size": 256,             # matches CDPM-Align (fidelity over T4 speed)
    # dataset-INDEX class conditioning (CDPM Sec. 3): 0=null/uncond token,
    # 1=Shenzhen chest, 2=ISBI2015 cephalometric, 3=DHA hand radiographs.
    # Phase-1 uses IMAGES only (DHA is unlabeled here; no DHA downstream eval).
    "num_classes": 4,
    "dataset_index": {"shenzhen": 1, "isbi2015": 2, "dha": 3},
    "pretrain_limit": None,        # cap images per source (None = use all)
    # Match CDPM-Align's pooled corpus SIZE exactly: 988 images across the three
    # datasets (Shenzhen+ISBI2015+DHA). Applied AFTER the no-leakage test
    # exclusion. The paper reports per-dataset TOTALS (Shenzhen 279, ISBI 400,
    # DHA 910) and the pooled total 988, but not the per-dataset test split, so
    # this is a self-consistent reconstruction: ISBI train=150 (standard 150/250
    # split), Shenzhen=279 (its full landmark set), DHA=559 (remainder to 988).
    # Keys are dataset INDICES (1=Shenzhen, 2=ISBI2015, 3=DHA). None = use all.
    "pretrain_targets": {1: 279, 2: 150, 3: 559},   # sums to 988 (CDPM parity)
    "n_shot": 25,                   # few-shot budget (paper: 10 / 25)
    "batch_size": 2,              # bigger 3-level+attn backbone runs the align loss as 4 fwd passes @256px; 2 fits T4 16GB (must match the DDPM run)
    "seed": 0,
    # generative pretraining -- CDPM-faithful ITERATION budget (updates), not epochs.
    # CDPM-Align: 50k updates @ effective batch 16, alignment only in the final
    # 10% of iterations. One T4 session at 256px caps ~5k updates (~0.59 s per
    # micro-batch), so we run a completable session at EFFECTIVE BATCH 16
    # (grad_accum) with the correct final-10% alignment schedule at images-seen
    # matched to v9; matching CDPM's full 800k-image budget needs multi-session
    # resume (see SUMMARY.md).
    # NOTE v17: alignment is turned back ON (align=True), so every step runs 4
    # backbone forwards (cond+uncond x two timesteps), NOT 2. Measured align-on
    # throughput is ~6.57 s/update (v10/v11 logs), ~2.2x the ~2.94 s/update
    # pure-flow rate (v15). We therefore REVERT pretrain_iters 6600 -> 3300 (the
    # v11 align-on-proven size). At 6.57 s/update: align_start=2970 is reached at
    # ~5.42 h and all 3300 iters finish at ~6.02 h, UNDER the 6.5 h soft cap --
    # so the final-10% alignment phase (iters 2970-3300) actually runs and the
    # aligned backbone is checkpointed for the downstream probes. Leaving
    # pretrain_iters=6600 here would repeat the v10 bug (cap fires before
    # align_start, alignment never runs).
    "pretrain_iters": 3300,          # align-on updates this session (~6.0h @256px T4)
    "grad_accum": 8,                 # micro-batches/update => eff batch = 2*8 = 16 (CDPM)
    "ema_decay": 0.0,               # EMA of backbone wts (e.g. 0.999). 0.0=OFF (disabled).
    "align_frac": 0.1,               # alignment active ONLY in the final 10% (iters 2970-3300)
    # PURE-FLOW ABLATION SWITCH (independent variable): False = flow-matching
    # loss ONLY (no multi-scale guidance alignment; ~2x faster but DIVERGES
    # from CDPM-Align). True = faithful CDPM-Align alignment.
    "align": True,
    # OT COUPLING (independent variable for this ablation): True = mini-batch
    # optimal-transport coupling computed over the FULL effective batch
    # (batch_size * grad_accum = 16). It re-orders the noise so each
    # (noise, image) pair minimises squared-L2 transport cost (Tong et al.
    # 2023, arXiv:2302.00482) -> straighter flow paths / fewer sampling steps.
    # False = independent coupling (standard CFM). Done at the effective-batch
    # level because a per-micro-batch OT over batch_size=2 would be a no-op.
    "ot_coupling": False,           # standard DDPM does not OT-couple its noise (faithful paper repro)
    # PROBABILITY PATH (independent variable): "linear" = straight-line
    # rectified-flow path; "schrodinger" = Schrodinger-bridge (Brownian-bridge)
    # path with diffusivity sb_sigma. With OT coupling on, a SMALL sb_sigma
    # approximates the true entropic Schrodinger bridge (unregularised OT is the
    # small-sigma limit; larger sigma injects more boundary-divergent drift that
    # the loss clamps saturate). sb_sigma=0 recovers the linear path.
    "interpolant": "schrodinger",
    "sb_sigma": 0.1,                 # bridge diffusivity (small: OT-consistent)
    "ckpt_every": 250,               # kill-safe checkpoint cadence (iterations)
    "epochs_pretrain": 80,           # legacy epoch budget (ignored when pretrain_iters is set)
    "max_pretrain_seconds": 6.5 * 3600,  # safety soft budget; won't bite (~6.0h expected), leaves ~2h for probes
    "lr_pretrain": 2e-4,
    "lambda_flow": 1.0,
    "lambda_align": 5.0,            # alignment strength (ablate in {0, 1, 5})
    "t_low": 0.25, "t_high": 0.75,  # mid-range timestep window (CDPM: [T/4, 3T/4])
    "feature_levels": ["enc_1_4", "enc_1_8", "bottleneck", "dec_1_8"],  # S
    "sigma": 3.0,
    # downstream probe
    "probe_epochs": 200,            # CDPM downstream epochs (was 50)
    "probe_lr": 1e-4,               # CDPM AdamW lr (was 1e-3)
    "probe_weight_decay": 1e-4,     # CDPM AdamW weight decay
    "probe_loss": "nll",            # CDPM spatial-softmax NLL (was heatmap MSE)
    "probe_patience": 15,           # CDPM early-stopping patience on test MRE
    "freeze_backbone": False,       # CDPM full fine-tune (was frozen linear probe)
    "test_frac": 0.3,

    # --- DEPENDENT variables (what we read out) -----------------------------
    "metrics": ["MRE", "P95", "SDR@2.0mm", "SDR@4.0mm"],

    # --- reference to beat (CDPM-Align, ISBI2015, 25-shot) ------------------
    "cdpm_ref": {"dataset": "ISBI2015", "n_shot": 25,
                 "MRE_mm": 1.54, "SDR@2mm": 77.52},
}

OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
E = SimpleNamespace(**EXPERIMENT)   # attribute access: E.image_size, E.n_shot, ...

def set_all_seeds(seed):
    """Pin RNGs so runs are reproducible (a controlled variable)."""
    random.seed(seed)
    try:
        import numpy as np; np.random.seed(seed)
    except Exception:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

set_all_seeds(E.seed)

RESULTS = []   # in-memory rows collected this session

def log_result(name, metrics, extra=None):
    """Record a run's dependent-variable readout together with the controlled
    config, print it, and append a row to OUT_DIR/experiment_results.csv so
    every comparison is traceable to the exact variables that produced it."""
    # This notebook runs the CFM *without* delta-alignment ablation, so record
    # the alignment state explicitly and report the *effective* lambda_align
    # (0 when alignment is off) instead of the unused EXPERIMENT default.
    align_on = bool(EXPERIMENT.get("align", True))
    row = {"when": datetime.datetime.now().isoformat(timespec="seconds"),
           "name": name, "backbone": EXPERIMENT["backbone"],
           **{k: EXPERIMENT[k] for k in ("dataset", "image_size", "n_shot", "seed")},
           "align": align_on,
           "lambda_align": (EXPERIMENT["lambda_align"] if align_on else 0.0),
           **(extra or {}),
           **{k: metrics.get(k) for k in EXPERIMENT["metrics"]}}
    RESULTS.append(row)
    print("[logged]", json.dumps(row, default=str))
    # protocol = how the downstream head was trained: "finetune" (backbone
    # updated) vs "linear-probe" (backbone frozen). init = backbone weights:
    # "pretrained" (CFM-pretrained) vs "random". Together they say exactly which
    # metric this row is (pretrained-finetune / random-finetune / frozen probe).
    cols = ["when", "name", "backbone", "dataset", "image_size", "n_shot",
            "align", "lambda_align", "seed", "probe", "protocol", "init", "unit",
            "MRE", "P95", "SDR@2.0mm", "SDR@4.0mm"]
    path = os.path.join(OUT_DIR, "experiment_results.csv")
    new = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow(row)

print("EXPERIMENT ready | independent=backbone:%s | controls: image_size=%d "
      "n_shot=%d lambda_align=%s seed=%d | metrics=%s"
      % (E.backbone, E.image_size, E.n_shot, E.lambda_align, E.seed, E.metrics))


## 0. Clone DeltaFlow & install dependencies

In [ ]:
import os, subprocess

# Idempotent + re-run safe: clone only if missing, then anchor cwd to the repo
# root via an ABSOLUTE path so re-running this cell never double-`cd`s (which
# was causing `FileNotFoundError` in the `%%writefile deltaflow/...` cells).
REPO = "/kaggle/working/deltaflow"
if not os.path.isdir(os.path.join(REPO, ".git")):
    rc = subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/phrugsa-limbunlom/deltaflow.git", REPO]
    ).returncode
    if rc != 0 or not os.path.isdir(REPO):
        raise RuntimeError(
            "git clone failed. Enable Internet in the Kaggle editor: right sidebar -> "
            "Notebook options -> Internet: On (account must be phone-verified), then "
            "re-run this cell."
        )
os.chdir(REPO)
print("cwd:", os.getcwd())

# Extra deps for data access + plotting (torch/pillow are already on Kaggle).
get_ipython().system('pip install -q kagglehub matplotlib pillow')

## 1. Inject the conditional UNet backbone

The multi-scale `ConditionalUNetVelocityField` is the piece that produces `(velocity, feature_dict)` for both a conditional and an unconditional pass — exactly what `DeltaAlignmentLoss` consumes. The next cell writes it into the package (`deltaflow/models/conditional_unet.py`), then re-installs so the import resolves.

In [ ]:
%%writefile deltaflow/models/conditional_unet.py
"""
A small multi-scale conditional UNet velocity field, built specifically to
plug into `DeltaAlignmentLoss` without pulling in a full diffusers/DiT
dependency. It produces `(velocity, feature_dict)` for a conditional and an
unconditional pass over real images.

Two conditioning mechanisms are supported (either or both):

* ``cond`` -- a spatial conditioning tensor (e.g. a landmark heatmap)
  concatenated onto the input as extra channels. ``cond=None`` zero-fills it.
* ``class_idx`` -- a per-sample dataset/class INDEX (CDPM-Align style, paper
  Sec. 2.2/3). Index ``0`` is the reserved unconditional/null token; real
  datasets start at ``1``. The index is embedded and broadcast as extra
  channels. ``class_idx=None`` falls back to the null token ``0``.

The unconditional pass (``cond=None`` and/or ``class_idx=0``) is the standard
classifier-free-guidance construction that ``delta_h = h_cond - h_uncond``
assumes.
"""

from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn

from ..core.base_velocity_field import BaseVelocityField


def _conv_block(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.GroupNorm(min(8, out_ch), out_ch),
        nn.SiLU(),
        nn.Conv2d(out_ch, out_ch, 3, padding=1),
        nn.GroupNorm(min(8, out_ch), out_ch),
        nn.SiLU(),
    )


class AttentionBlock(nn.Module):
    """Multi-head self-attention over the H*W spatial grid (guided-diffusion
    style), applied at the low-resolution bottleneck so the receptive field is
    global. Residual: ``x + proj(attn(norm(x)))``. CDPM-Align's DDPM backbone
    uses attention at the coarse scales; adding it here closes part of the
    capacity gap vs the original 2-level, attention-free UNet."""

    def __init__(self, channels: int, num_heads: int = 4):
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"
        self.num_heads = num_heads
        self.norm = nn.GroupNorm(min(8, channels), channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        nh, cph = self.num_heads, C // self.num_heads
        qkv = self.qkv(self.norm(x)).reshape(B, 3, nh, cph, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.softmax(
            torch.einsum("bhci,bhcj->bhij", q, k) / (cph ** 0.5), dim=-1
        )
        out = torch.einsum("bhij,bhcj->bhci", attn, v).reshape(B, C, H, W)
        return x + self.proj(out)


class ConditionalUNetVelocityField(BaseVelocityField):
    """Multi-scale conditional velocity field with intermediate feature access.

    Three-level encoder/decoder (H -> H/2 -> H/4 -> H/8) with a self-attention
    block at the bottleneck. ``feature_channels()`` reports the channel count of
    each exposed feature map so downstream consumers (the multi-scale projector
    and the probe head) never hard-code dims that could drift from the backbone.
    """

    FEATURE_KEYS = ("enc_1_4", "enc_1_8", "bottleneck", "dec_1_8")

    def __init__(
        self,
        in_channels: int = 1,
        cond_channels: int = 1,
        base_channels: int = 64,
        time_dim: int = 64,
        num_classes: int = 0,
        class_dim: int = 32,
        attention_heads: int = 4,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.cond_channels = cond_channels
        self.num_classes = num_classes
        self.class_dim = class_dim if num_classes > 0 else 0
        c = base_channels
        self.base_channels = c

        self.time_embed = nn.Sequential(
            nn.Linear(1, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim)
        )

        # CDPM-Align dataset/class conditioning: an embedding table whose row 0
        # is the unconditional/null token. Broadcast to a feature map and
        # concatenated alongside the (optional) spatial cond + time channels.
        if num_classes > 0:
            self.class_embed = nn.Embedding(num_classes, class_dim)

        total_in = in_channels + cond_channels + time_dim + self.class_dim
        self.stem = nn.Conv2d(total_in, c, 3, padding=1)

        # Encoder: three downsampling stages.
        self.enc1 = _conv_block(c, c)                              # @H    , c
        self.down1 = nn.Conv2d(c, c, 3, stride=2, padding=1)
        self.enc2 = _conv_block(c, c * 2)                          # @H/2  , 2c
        self.down2 = nn.Conv2d(c * 2, c * 2, 3, stride=2, padding=1)
        self.enc3 = _conv_block(c * 2, c * 4)                      # @H/4  , 4c
        self.down3 = nn.Conv2d(c * 4, c * 4, 3, stride=2, padding=1)
        self.bottleneck = _conv_block(c * 4, c * 8)                # @H/8  , 8c
        self.bottleneck_attn = AttentionBlock(c * 8, num_heads=attention_heads)

        # Decoder: mirror the encoder with skip connections.
        self.up1 = nn.ConvTranspose2d(c * 8, c * 4, 2, stride=2)
        self.dec1 = _conv_block(c * 8, c * 4)                      # @H/4  , 4c
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, 2, stride=2)
        self.dec2 = _conv_block(c * 4, c * 2)                      # @H/2  , 2c
        self.up3 = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.dec3 = _conv_block(c * 2, c)                          # @H    , c

        self.out = nn.Conv2d(c, in_channels, 3, padding=1)

    def feature_channels(self) -> Dict[str, int]:
        """Channel count of each feature map returned by
        ``forward_with_features`` -- the single source of truth for downstream
        head/projector input dims."""
        c = self.base_channels
        return {"enc_1_4": c, "enc_1_8": c * 2, "bottleneck": c * 8, "dec_1_8": c * 2}

    def _embed_time(self, x: torch.Tensor, t) -> torch.Tensor:
        if not isinstance(t, torch.Tensor):
            t = torch.tensor(t, dtype=x.dtype, device=x.device)
        if t.dim() == 0:
            t = t.expand(x.shape[0])
        t = t.to(device=x.device, dtype=x.dtype)
        emb = self.time_embed(t.view(-1, 1))
        return emb.view(*emb.shape, 1, 1).expand(-1, -1, x.shape[2], x.shape[3])

    def _embed_class(self, x: torch.Tensor, class_idx) -> torch.Tensor:
        """Embed a per-sample class index (``None`` -> null token 0) and
        broadcast it to an ``(B, class_dim, H, W)`` feature map."""
        if class_idx is None:
            class_idx = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)
        elif not isinstance(class_idx, torch.Tensor):
            class_idx = torch.tensor(class_idx, device=x.device)
        class_idx = class_idx.to(device=x.device, dtype=torch.long).view(-1)
        if class_idx.shape[0] == 1 and x.shape[0] > 1:
            class_idx = class_idx.expand(x.shape[0])
        emb = self.class_embed(class_idx)  # (B, class_dim)
        return emb.view(emb.shape[0], emb.shape[1], 1, 1).expand(
            -1, -1, x.shape[2], x.shape[3]
        )

    def forward_with_features(
        self,
        x: torch.Tensor,
        t,
        cond: Optional[torch.Tensor] = None,
        class_idx=None,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """Return ``(velocity, feats)``. For the UNCONDITIONAL pass use
        ``cond=None`` (spatial cond zero-filled) and/or ``class_idx=0`` (null
        class token)."""
        t_emb = self._embed_time(x, t)

        parts = [x]
        if self.cond_channels > 0:
            if cond is None:
                cond = torch.zeros(
                    x.shape[0], self.cond_channels, x.shape[2], x.shape[3],
                    device=x.device, dtype=x.dtype,
                )
            parts.append(cond)
        parts.append(t_emb)
        if self.num_classes > 0:
            parts.append(self._embed_class(x, class_idx))
        h = torch.cat(parts, dim=1)

        h = self.stem(h)

        e1 = self.enc1(h)                       # @H   , c
        e2 = self.enc2(self.down1(e1))          # @H/2 , 2c
        e3 = self.enc3(self.down2(e2))          # @H/4 , 4c
        bott = self.bottleneck(self.down3(e3))  # @H/8 , 8c
        bott = self.bottleneck_attn(bott)

        u1 = self.up1(bott)                     # @H/4 , 4c
        dec1 = self.dec1(torch.cat([u1, e3], dim=1))   # @H/4 , 4c
        u2 = self.up2(dec1)                     # @H/2 , 2c
        dec2 = self.dec2(torch.cat([u2, e2], dim=1))   # @H/2 , 2c
        u3 = self.up3(dec2)                     # @H   , c
        dec3 = self.dec3(torch.cat([u3, e1], dim=1))   # @H   , c

        velocity = self.out(dec3)

        feats = {
            "enc_1_4": e1,
            "enc_1_8": e2,
            "bottleneck": bott,
            "dec_1_8": dec2,
        }
        return velocity, feats

    def forward(self, x: torch.Tensor, t, **cond) -> torch.Tensor:
        velocity, _ = self.forward_with_features(
            x, t, cond=cond.get("cond"), class_idx=cond.get("class_idx")
        )
        return velocity


__all__ = ["ConditionalUNetVelocityField"]


In [ ]:
# Editable install so `deltaflow.models.conditional_unet` (just written) is importable.
!pip install -e . -q
import deltaflow, deltaflow.models.conditional_unet  # sanity check
print("deltaflow ready:", deltaflow.__file__)

## 2. Get the datasets

Phase-1 CFM pretraining pools three radiograph sources into one corpus, each tagged with a **dataset-index class token** (CDPM-Align-style index conditioning that needs zero manual annotation): `0 = null/uncond`, `1 = Shenzhen`, `2 = ISBI2015`, `3 = DHA` (`num_classes = 4`). The images are used **as-is** for pretraining; landmark heads are added only later by the downstream probe.

**Data sources**

| Idx | Dataset | Modality | Kaggle source |
|----:|---------|----------|---------------|
| 1 | Shenzhen TB set | Chest X-ray | [`raddar/tuberculosis-chest-xrays-shenzhen`](https://www.kaggle.com/datasets/raddar/tuberculosis-chest-xrays-shenzhen) |
| 2 | ISBI2015 | Cephalometric | [`jiahongqian/cephalometric-landmarks`](https://www.kaggle.com/datasets/jiahongqian/cephalometric-landmarks) |
| 3 | Digital Hand Atlas | Hand X-ray | [`phrugsalimbunlom/digital-hand-atlas-256`](https://www.kaggle.com/datasets/phrugsalimbunlom/digital-hand-atlas-256) |

The Shenzhen and DHA sets ship **images only** (no landmarks); ISBI2015 additionally supplies the `*_senior.csv` landmarks used for the millimetre downstream evaluation in §6.5. The pretraining corpus **holds out every downstream test image** to avoid leakage.

The next cell locates each source whether you attached it via Kaggle **Add Input** or let `kagglehub` download it; missing optional sources are skipped so single-dataset runs still work.

> *Real anatomical landmarks* for Shenzhen are wired up in **§6**: it clones [`ngaggion/Chest-xray-landmark-dataset`](https://github.com/ngaggion/Chest-xray-landmark-dataset) (RL + LL lung landmarks), aligns them to the images, and re-runs the probe with `dataset="chest"`.

In [ ]:
import os, glob

def find_shenzhen_images():
    # 1) Kaggle "Add Input" mount (no download).
    for pat in ("/kaggle/input/**/CXR_png", "/kaggle/input/**/images"):
        hits = glob.glob(pat, recursive=True)
        hits = [h for h in hits if glob.glob(os.path.join(h, "*.png"))]
        if hits:
            return hits[0]
    # 2) Fall back to kagglehub download (Kaggle/Colab both work).
    import kagglehub
    path = kagglehub.dataset_download("raddar/tuberculosis-chest-xrays-shenzhen")
    pngs = glob.glob(os.path.join(path, "**", "*.png"), recursive=True)
    if not pngs:
        raise FileNotFoundError(f"No .png found under {path}")
    return os.path.dirname(sorted(pngs)[0])

ROOT = find_shenzhen_images()
n_png = len(glob.glob(os.path.join(ROOT, "*.png")))
print("images dir:", ROOT)
print("num pngs  :", n_png)
assert n_png > 0, "No PNGs found -- check the dataset path."


def find_isbi_images():
    """Locate the ISBI2015 cephalometric jpg dir (dataset index 2). Returns the
    directory DIRECTLY containing the .jpg images, or None if unavailable so
    single-dataset runs still work."""
    for pat in ("/kaggle/input/**/cepha400", "/kaggle/input/**/RawImage",
                "/kaggle/input/**/*Image*", "/kaggle/input/**"):
        for h in glob.glob(pat, recursive=True):
            if os.path.isdir(h) and glob.glob(os.path.join(h, "*.jpg")):
                return h
    try:
        import kagglehub
        path = kagglehub.dataset_download("jiahongqian/cephalometric-landmarks")
        jpgs = glob.glob(os.path.join(path, "**", "*.jpg"), recursive=True)
        if jpgs:
            return os.path.dirname(sorted(jpgs)[0])
    except Exception as e:
        print("[warn] ISBI2015 kagglehub download failed:", e)
    return None


ISBI_ROOT = find_isbi_images()
n_isbi = len(glob.glob(os.path.join(ISBI_ROOT, "*.jpg"))) if ISBI_ROOT else 0
print("isbi dir  :", ISBI_ROOT)
print("num jpgs  :", n_isbi)


def find_isbi_csv_dir():
    """Locate the dir holding the ISBI2015 *_senior.csv landmark files."""
    roots = ["/kaggle/input"]
    if ISBI_ROOT:
        roots += [ISBI_ROOT, os.path.dirname(ISBI_ROOT),
                  os.path.dirname(os.path.dirname(ISBI_ROOT))]
    for r in roots:
        hits = glob.glob(os.path.join(r, "**", "*_senior.csv"), recursive=True)
        if hits:
            return os.path.dirname(sorted(hits)[0])
    return None


ISBI_CSV_DIR = find_isbi_csv_dir()
print("isbi csv  :", ISBI_CSV_DIR)


def find_dha_images():
    """Locate the Digital Hand Atlas jpg dir (dataset index 3). Returns the dir
    DIRECTLY containing the .jpg images, or None if the dataset is not attached
    so Shenzhen+ISBI-only runs still work. The Kaggle dataset
    `phrugsalimbunlom/digital-hand-atlas-256` mounts its 256px images under
    /kaggle/input/digital-hand-atlas-256/images."""
    for pat in ("/kaggle/input/**/digital-hand-atlas*/**",
                "/kaggle/input/**/digital-hand-atlas*",
                "/kaggle/input/**"):
        for h in glob.glob(pat, recursive=True):
            if os.path.isdir(h) and glob.glob(os.path.join(h, "*.jpg")) \
               and ("hand" in h.lower() or "dha" in h.lower()):
                return h
    try:
        import kagglehub
        path = kagglehub.dataset_download("phrugsalimbunlom/digital-hand-atlas-256")
        jpgs = glob.glob(os.path.join(path, "**", "*.jpg"), recursive=True)
        if jpgs:
            return os.path.dirname(sorted(jpgs)[0])
    except Exception as e:
        print("[warn] DHA download failed:", e)
    return None


DHA_ROOT = find_dha_images()
n_dha = len(glob.glob(os.path.join(DHA_ROOT, "*.jpg"))) if DHA_ROOT else 0
print("dha dir   :", DHA_ROOT)
print("num dha   :", n_dha)

## 3. Write & run the CFM + delta-alignment training module

This wires `LinearInterpolant` (rectified-flow path) + `ConditionalUNetVelocityField` together (with the multi-scale `DeltaAlignmentLoss` applied only in the final `align_frac` of iterations) — the CFM backbone swap for CDPM-Align's multi-scale guidance alignment. The current run sets `EXPERIMENT["align"]=False` (**pure flow-matching**: 2 forward passes/step and only `loss_flow` logged); the corpus pools **Shenzhen + ISBI2015** with downstream **test images excluded** (no leakage). The next cell writes the training module, then the one after runs it and returns the loss history.

In [ ]:
%%writefile cfm_pretrain.py
"""
20-training/03-cfm-delta-align-pretrain: real multi-scale guidance-aligned
CFM pretraining -- the CDPM-Align backbone swap (DDPM -> flow matching),
using ConditionalUNetVelocityField + DeltaAlignmentLoss on an actual
radiograph dataset.

This is the "one experiment you can run now" from the DPhil application
plan: same multi-scale guidance-alignment mechanism as CDPM-Align, but the
generative backbone is trained with a flow-matching objective (continuous
t, LinearInterpolant) instead of DDPM noise prediction.

Usage (from repo root, after `pip install -e .`):

    # Combined dataset-index pretraining (default): Shenzhen chest X-rays +
    # ISBI2015 cephalograms with dataset-index conditioning.
    python pretrain_cfm_delta_align.py \\
        --dataset combined --root /path/to/data \\
        --chest-root /path/to/shenzhen/images \\
        --isbi-root /path/to/isbi2015/images \\
        --num-classes 3 --image-size 256 --batch-size 4 --epochs 30

    # Real anatomical landmarks (optional), e.g. ngaggion lung landmarks:
    python pretrain_cfm_delta_align.py \\
        --dataset chest --root /path/to/shenzhen/images \\
        --landmarks-dir /path/to/landmarks-npy \\
        --image-size 128 --batch-size 8 --epochs 30

    # Cephalometric benchmark (needs deltaflow/datasets/isbi2015.py):
    python pretrain_cfm_delta_align.py \\
        --dataset isbi2015 --root /path/to/isbi2015 --split train

Runs on CPU (slow) or CUDA automatically. Designed to fit on a free-tier
Kaggle/Colab GPU for a few-hundred-image dataset at image-size 128-256.

Importable: call ``run_training(args)`` from a notebook to get the loss
history back for plotting; ``main()`` is the CLI entry point.
"""

import argparse
import glob
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader

from deltaflow.datasets import ChestXrayDataset
from deltaflow.interpolants import LinearInterpolant, SchrodingerBridgeInterpolant
from deltaflow.losses.delta_alignment import DeltaAlignmentLoss, flow_matching_velocity_loss
from deltaflow.models import MultiScaleProjector
from deltaflow.models.conditional_unet import ConditionalUNetVelocityField
from deltaflow.trainer.coupling import OTCoupling


# --------------------------------------------------------------------------
# Landmark -> heatmap conditioning signal
# --------------------------------------------------------------------------

def landmarks_to_heatmap(
    landmarks: torch.Tensor, image_size: int, sigma: float = 4.0
) -> torch.Tensor:
    """Render (N, 2) pixel-space landmarks as a single-channel Gaussian
    heatmap, summed over all landmarks and clamped to [0, 1]. This is the
    conditioning signal fed to the "conditional" pass; the "unconditional"
    pass zeroes this channel (see ConditionalUNetVelocityField)."""
    device = landmarks.device
    ys = torch.arange(image_size, device=device).view(1, -1, 1)
    xs = torch.arange(image_size, device=device).view(1, 1, -1)
    heat = torch.zeros(1, image_size, image_size, device=device)
    for x, y in landmarks:
        heat = heat + torch.exp(-((xs - x) ** 2 + (ys - y) ** 2) / (2 * sigma**2))
    return heat.clamp(0.0, 1.0)


# --------------------------------------------------------------------------
# Dataset construction
# --------------------------------------------------------------------------

class CombinedImageDataset(torch.utils.data.Dataset):
    """Pool multiple datasets' images for Phase-1 CFM pretraining (CDPM-Align
    combined-corpus setup). Each item is ``(image[1,H,W], dataset_index)``.

    Dataset index ``0`` is RESERVED as the unconditional/null token; real
    datasets are numbered from ``1`` (e.g. 1=Shenzhen chest, 2=ISBI2015
    cephalometric). This matches CDPM's dataset-INDEX class conditioning:
    Phase-1 pretraining needs only IMAGES, no landmarks.
    """

    def __init__(self, sources, image_size=256, per_source_limit=None,
                 exclude_stems=None, per_source_targets=None):
        # sources: list of (root_dir, dataset_index, glob_pattern)
        # per_source_targets: optional {dataset_index: max_kept} cap applied
        #   AFTER the no-leakage test exclusion, to match CDPM-Align's pooled
        #   corpus SIZE (988 images across the three datasets). Our raw sources
        #   are larger than CDPM's (Shenzhen 662 vs 279, DHA 1390 vs 910), so
        #   without this the corpus would exceed 988 and corpus-size would be an
        #   uncontrolled variable vs the paper. Truncation is deterministic
        #   (sorted-filename prefix) => reproducible.
        # exclude_stems: set of file STEMS (basename without extension) to drop
        #   from the pooled corpus. Used to keep every DOWNSTREAM test image OUT
        #   of pretraining (CDPM-Align pools "the ... corpus, excluding the test
        #   set"); pretraining on eval images would be self-supervised leakage
        #   that inflates our numbers vs the paper. Stems are unambiguous across
        #   datasets here (ISBI '150' vs Shenzhen 'CHNCXR_0001_0').
        self.image_size = image_size
        exclude_stems = set(exclude_stems or ())
        self.items = []  # (path, dataset_index)
        n_excluded = 0
        for root, didx, pat in sources:
            if not root:
                continue
            paths = sorted(glob.glob(os.path.join(root, pat)))
            if exclude_stems:
                kept = [p for p in paths
                        if os.path.splitext(os.path.basename(p))[0] not in exclude_stems]
                n_excluded += len(paths) - len(kept)
                paths = kept
            if per_source_limit is not None:
                paths = paths[:per_source_limit]
            if per_source_targets and didx in per_source_targets:
                paths = paths[:per_source_targets[didx]]
            self.items += [(p, didx) for p in paths]
        if exclude_stems:
            print(f"CombinedImageDataset: excluded {n_excluded} downstream test "
                  f"images from the pretraining corpus (no-leakage).")
        if not self.items:
            raise RuntimeError(
                "CombinedImageDataset found no images -- check the source roots."
            )
        counts = {}
        for _, didx in self.items:
            counts[didx] = counts.get(didx, 0) + 1
        print(f"CombinedImageDataset: {len(self.items)} images by class index {counts}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, didx = self.items[i]
        img = Image.open(path).convert("L").resize(
            (self.image_size, self.image_size), Image.BILINEAR
        )
        x = torch.from_numpy(np.asarray(img, dtype="float32") / 255.0).unsqueeze(0)
        return x, didx


class ISBI2015LandmarkDataset(torch.utils.data.Dataset):
    """ISBI2015 cephalometric landmark dataset for the DOWNSTREAM few-shot probe,
    faithful to CDPM-Align (``datasets/cephalo_dataset.py``).

    * 19 landmarks; native pixel spacing 0.1 mm.
    * Split is by SORTED filename (001..400), NOT the official challenge split:
      ``train=[:130]``, ``val=[130:150]``, ``train_val=[:150]``,
      ``test=[150:400]`` (250 eval images), ``all``.
    * Error is measured in ``image_size`` px then converted to mm per axis via
      ``mm/px = (native_dim / image_size) * 0.1`` (see ``mm_per_pixel``, a
      length-2 ``[x, y]`` vector).

    NOTE: the public Kaggle mirror ``jiahongqian/cephalometric-landmarks`` ships
    SENIOR annotations only (``*_senior.csv`` with columns
    ``image_path,1_x,1_y,...,19_x,19_y``). CDPM averages junior+senior; using
    senior-only is a small, documented deviation.
    """

    NATIVE_MM_PER_PX = 0.1
    NUM_LANDMARKS = 19
    _SPLITS = {
        "train": slice(0, 130), "val": slice(130, 150),
        "train_val": slice(0, 150), "test": slice(150, 400), "all": slice(0, None),
    }

    def __init__(self, image_dir, csv_dir, image_size=128, phase="all"):
        import csv as _csv
        if phase not in self._SPLITS:
            raise ValueError(f"Unknown ISBI2015 phase {phase!r}")
        self.image_size = image_size

        self._img = {}  # '001.jpg' -> full path
        for p in glob.glob(os.path.join(image_dir, "**", "*.jpg"), recursive=True):
            self._img[os.path.basename(p)] = p

        self._lm = {}  # '001.jpg' -> (19, 2) native-px landmarks
        csvs = sorted(glob.glob(os.path.join(csv_dir, "**", "*_senior.csv"),
                                recursive=True))
        if not csvs:
            raise RuntimeError(f"No *_senior.csv landmark files under {csv_dir!r}")
        for cf in csvs:
            with open(cf) as fh:
                reader = _csv.reader(fh)
                next(reader, None)  # header
                for row in reader:
                    if not row:
                        continue
                    vals = [float(v) for v in row[1:1 + 2 * self.NUM_LANDMARKS]]
                    self._lm[row[0]] = np.array(vals, dtype="float32").reshape(-1, 2)

        names = sorted(n for n in self._lm if n in self._img)
        if len(names) < 400:
            print(f"[warn] ISBI2015: only {len(names)} image+landmark pairs matched "
                  f"(expected 400) -- check image_dir/csv_dir mounts.")
        self.names = names[self._SPLITS[phase]]

        h, w = self._native_hw()
        self.native_size = (w, h)
        self.mm_per_pixel = [
            (w / image_size) * self.NATIVE_MM_PER_PX,
            (h / image_size) * self.NATIVE_MM_PER_PX,
        ]
        print(f"ISBI2015[{phase}]: {len(self.names)} imgs | native {w}x{h} | "
              f"mm/px @{image_size} = {[round(m, 4) for m in self.mm_per_pixel]}")

    def _native_hw(self):
        name = self.names[0] if self.names else next(iter(self._img))
        with Image.open(self._img[name]) as im:
            w, h = im.size
        return h, w

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        name = self.names[i]
        with Image.open(self._img[name]) as im:
            w, h = im.size
            img = im.convert("L").resize(
                (self.image_size, self.image_size), Image.BILINEAR
            )
        x = torch.from_numpy(np.asarray(img, dtype="float32") / 255.0).unsqueeze(0)
        lm = self._lm[name].copy()
        lm[:, 0] *= self.image_size / w
        lm[:, 1] *= self.image_size / h
        return x, torch.from_numpy(lm)


class ChestLandmarkDataset(ChestXrayDataset):
    def __init__(self, root, landmarks_dir, image_size=None, n_shot=None,
                 ref_size=1024):
        self._landmarks_dir = Path(landmarks_dir)
        self._ref_size = ref_size
        self._target_size = image_size
        # normalize_landmarks=False: keep pixel-space (not [-1, 1])
        # coordinates, since the downstream probe (heatmap targets,
        # mm_per_pixel metrics) expects pixel units.
        super().__init__(root=root, image_size=image_size,
                         landmarks_file="dummy", normalize_landmarks=False)
        # Keep only images that actually have a landmark file, so the
        # base __getitem__ never indexes a missing key. Sort by stem so
        # the ordering (and therefore the seeded train/test split) is
        # DETERMINISTIC and independent of the base class's glob order --
        # this is what lets pretraining reproduce the exact downstream
        # test set to hold it out of the corpus (no leakage).
        self.image_paths = sorted(
            (p for p in self.image_paths if p.stem in self.landmarks),
            key=lambda p: p.stem,
        )
        if not self.image_paths:
            raise RuntimeError(
                f"No image matched a landmark .npy in {self._landmarks_dir}"
            )
        if n_shot is not None:
            self.image_paths = self.image_paths[:n_shot]

    def _load_landmarks(self, _):
        # RadiographDataset.__getitem__ rescales landmarks from the
        # ORIGINAL (pre-resize) on-disk image pixel grid to image_size
        # via (image_size / orig_w, image_size / orig_h). So here we
        # must convert the annotations from the ref_size frame to the
        # actual on-disk pixel grid (ref_size -> orig_w/orig_h), NOT to
        # image_size directly -- otherwise the base class's own rescale
        # would apply a second, compounding scale factor.
        out = {}
        for p in self.image_paths:
            npy_path = self._landmarks_dir / f"{p.stem}.npy"
            if npy_path.exists():
                coords = np.load(npy_path).reshape(-1, 2).astype("float32")
                with Image.open(p) as im:
                    orig_w, orig_h = im.size
                scale = np.array(
                    [orig_w / self._ref_size, orig_h / self._ref_size],
                    dtype="float32",
                )
                out[p.stem] = torch.tensor(coords * scale)
        return out

    def __getitem__(self, idx):
        # RadiographDataset flattens landmarks to (K*2,); restore the
        # (K, 2) shape the downstream probe expects.
        x, lm = super().__getitem__(idx)
        return x, lm.view(-1, 2)


def _held_out_split_indices(n, test_frac, seed):
    """Reproduce the DOWNSTREAM probe's deterministic test split (see
    ``probe_landmark_detection._split_indices``): a seeded permutation whose
    first ``round(n*test_frac)`` entries are the fixed held-out test set. We only
    need the test indices here (to exclude those images from pretraining), so the
    n_shot/train logic is intentionally omitted. Kept byte-for-byte compatible
    with the probe's generator so the two agree on which images are 'test'."""
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g).tolist()
    test_size = max(1, int(round(n * test_frac)))
    return perm[:test_size]


def isbi_test_stems(isbi_root):
    """Stems of the ISBI2015 DOWNSTREAM test split (CDPM sorted-filename
    ``test=[150:400]``), so pretraining can hold them out. Matches
    ``ISBI2015LandmarkDataset`` (which slices sorted basenames [150:400])."""
    names = sorted(os.path.basename(p)
                   for p in glob.glob(os.path.join(isbi_root, "*.jpg")))
    test = names[ISBI2015LandmarkDataset._SPLITS["test"]]
    return {os.path.splitext(n)[0] for n in test}


def chest_test_stems(chest_pp_root, landmarks_dir, test_frac, seed):
    """Stems of the Shenzhen chest DOWNSTREAM test split, reproducing the probe's
    seeded ``_split_indices`` over the SAME deterministic (stem-sorted) list of
    landmark-having images, so pretraining can hold those test images out."""
    lm_dir = Path(landmarks_dir)
    stems = sorted(
        os.path.splitext(os.path.basename(p))[0]
        for p in glob.glob(os.path.join(chest_pp_root, "*.png"))
        if (lm_dir / f"{os.path.splitext(os.path.basename(p))[0]}.npy").exists()
    )
    if not stems:
        return set()
    test_idx = _held_out_split_indices(len(stems), test_frac, seed)
    return {stems[i] for i in test_idx}


def combined_test_stems(args):
    """Union of every downstream test-image stem for the combined corpus, so
    ``build_dataset`` can pass it to ``CombinedImageDataset`` as ``exclude_stems``
    and keep the pretraining corpus test-free (CDPM-Align: 'excluding the test
    set'). Controlled by ``args.exclude_test`` (default True)."""
    stems = set()
    if getattr(args, "isbi_root", None):
        stems |= isbi_test_stems(args.isbi_root)
    pp_root = getattr(args, "chest_pp_root", None)
    lm_dir = getattr(args, "chest_landmarks_dir", None)
    if pp_root and lm_dir:
        stems |= chest_test_stems(
            pp_root, lm_dir,
            test_frac=getattr(args, "test_frac", 0.2),
            seed=getattr(args, "seed", 0),
        )
    return stems


def build_dataset(args):
    if args.dataset == "isbi2015":
        # DOWNSTREAM fine-tune/eval target, faithful to CDPM-Align's cephalo
        # protocol (datasets/cephalo_dataset.py): 19 landmarks, sorted-filename
        # split train=[:130] / val=[130:150] / test=[150:400], 0.1 mm/px native.
        return ISBI2015LandmarkDataset(
            image_dir=args.root,
            csv_dir=getattr(args, "landmarks_dir", None) or args.root,
            image_size=args.image_size,
            phase=getattr(args, "phase", "all"),
        )
    if args.dataset == "combined":
        # CDPM-Align faithful Phase-1 corpus: pooled images, each tagged with a
        # dataset index for class conditioning. Uses ALL images (no n_shot cap;
        # n_shot is a DOWNSTREAM few-shot budget, not a pretraining cap) EXCEPT
        # the downstream test images, which are held out to avoid leakage --
        # CDPM-Align pools "the ... pretraining corpus, excluding the test set".
        sources = []
        if getattr(args, "chest_root", None):
            sources.append((args.chest_root, 1, "*.png"))
        if getattr(args, "isbi_root", None):
            sources.append((args.isbi_root, 2, "*.jpg"))
        # DHA hand radiographs (dataset index 3) -- completes CDPM-Align's
        # 3-dataset pooled corpus (Shenzhen + ISBI2015 + DHA). Unlabeled; used
        # for Phase-1 pretraining only (we hold no DHA downstream test split, so
        # every DHA image is safe to include).
        if getattr(args, "dha_root", None):
            sources.append((args.dha_root, 3, "*.jpg"))
        exclude_stems = (
            combined_test_stems(args)
            if getattr(args, "exclude_test", True) else set()
        )
        return CombinedImageDataset(
            sources, image_size=args.image_size,
            per_source_limit=getattr(args, "pretrain_limit", None),
            exclude_stems=exclude_stems,
            per_source_targets=getattr(args, "pretrain_targets", None),
        )
    if args.dataset == "chest":
        # Real anatomical landmarks: expects per-image `.npy` files of shape
        # (N, 2) named `<image_stem>.npy` in --landmarks-dir. Coordinates are
        # scaled from --landmark-ref-size to --image-size (adjust ref-size to
        # the resolution the landmarks were annotated at).
        if args.landmarks_dir is None:
            raise ValueError("--landmarks-dir is required for --dataset chest")
        return ChestLandmarkDataset(
            args.root, args.landmarks_dir, image_size=args.image_size,
            n_shot=args.n_shot, ref_size=args.landmark_ref_size,
        )
    else:
        raise ValueError(f"Unknown --dataset {args.dataset!r}")


# --------------------------------------------------------------------------
# Training loop
# --------------------------------------------------------------------------

class DDPMInterpolant:
    """DDPM (discrete variance-preserving) forward process with a NOISE target.

    Faithful reproduction of CDPM-Align's DDPM noise-prediction backbone, used
    as the internal baseline for the iso-compute CFM-vs-DDPM head-to-head. It
    exposes the SAME ``interpolate(x1, t, x0=None) -> (x_t, target)`` interface
    as the flow-matching interpolants, so the entire rest of the pipeline -- the
    identical UNet, the multi-scale delta-alignment loss, the two-timestep
    machinery and the OT coupling -- is reused UNCHANGED. The ONLY things that
    differ from the CFM path are (a) the forward corruption process and (b) the
    regression target (velocity -> noise epsilon). That single swap IS the
    independent variable of the head-to-head.

    Schedule (paper): linear beta from ``beta_start=1e-4`` to ``beta_end=0.028``
    over ``num_steps=500`` steps; ``alpha_bar_i = prod_{k<=i}(1 - beta_k)``.

    Convention: this repo's interpolants use ``t=1 -> clean data x1`` and
    ``t=0 -> pure noise x0``. We keep that convention by mapping continuous
    ``t in [0, 1]`` to the discrete DDPM index ``i = round((1 - t)*(T - 1))``,
    so ``t=1`` selects the cleanest step (alpha_bar ~ 1) and ``t=0`` the
    noisiest (alpha_bar small). The CDPM feature-extraction step ``t=200`` (of
    ``T=500``) therefore corresponds to ``t ~ 1 - 200/499 ~ 0.6`` on this
    continuous axis (see the probe's ``t_value`` for the DDPM backbone).

    Forward / target::

        x_t    = sqrt(alpha_bar) * x1 + sqrt(1 - alpha_bar) * x0   (x0 == eps)
        target = x0                                                (predict eps)

    ``x0`` is threaded through so OT coupling still works (it couples which
    noise pairs with which image); with ``x0=None`` each call draws independent
    Gaussian noise -- i.e. standard DDPM. For a faithful paper reproduction run
    with ``ot_coupling=False`` (standard DDPM does not OT-couple its noise).
    """

    def __init__(self, num_steps: int = 500,
                 beta_start: float = 1e-4, beta_end: float = 0.028):
        self.num_steps = int(num_steps)
        betas = torch.linspace(beta_start, beta_end, self.num_steps)
        self.alpha_bars = torch.cumprod(1.0 - betas, dim=0)  # (T,), decreasing

    def interpolate(self, x1, t, x0=None):
        if x0 is None:
            x0 = torch.randn_like(x1)  # x0 == the target noise epsilon
        idx = torch.round((1.0 - t) * (self.num_steps - 1)).long()
        idx = idx.clamp(0, self.num_steps - 1)
        ab = self.alpha_bars.to(device=x1.device, dtype=x1.dtype)[idx]
        ab = ab.view(-1, *([1] * (x1.dim() - 1)))
        x_t = torch.sqrt(ab) * x1 + torch.sqrt(1.0 - ab) * x0
        target = x0
        return x_t, target


def _compute_batch_loss(backbone, loss_fn, interpolant, x1, labels, args, device,
                        x0_1=None, x0_2=None):
    """One micro-batch forward pass -> (total_loss, loss_dict).

    Shared by both the legacy epoch loop and the CDPM-faithful iteration-budget
    loop so the two paths compute *identical* losses. See the long comment in
    ``run_training`` for the CDPM-Align rationale (two-timestep alignment,
    dataset-index conditioning, full-range [0, 1] flow-matching timesteps).

    ``x0_1``/``x0_2`` optionally supply PRE-COUPLED noise for the two alignment
    timesteps (used by the mini-batch OT coupling, which is computed over the
    full effective batch in ``run_training``). When ``None`` the interpolant
    draws its own independent noise, i.e. standard independent CFM coupling.
    """
    x1 = x1.to(device)  # (B, 1, H, W)
    labels = labels.to(device).long()  # dataset index per sample (>=1)
    B = x1.shape[0]
    null = torch.zeros_like(labels)  # class 0 == reserved unconditional token

    if getattr(args, "align_t_window", False):
        span = args.t_high - args.t_low
        t1 = args.t_low + span * torch.rand(B, device=device)
    else:
        t1 = torch.rand(B, device=device)

    # ------------------------------------------------------------------
    # PURE-FLOW ABLATION (args.align is False): train the flow-matching
    # velocity objective ONLY -- no multi-scale guidance-delta alignment.
    # This drops the second-timestep forward pair AND the projector, so each
    # step runs 2 backbone forwards (conditional + unconditional) instead of 4,
    # roughly halving pretraining compute. Both branches are still trained so
    # classifier-free guidance is learned throughout.
    # NOTE: this DIVERGES from CDPM-Align -- the alignment mechanism the paper
    # credits for its few-shot gains is absent; treat any result as a
    # "CFM without alignment" ablation, NOT a CDPM-Align reproduction.
    # ------------------------------------------------------------------
    if not getattr(args, "align", True):
        x_t1, target_1 = interpolant.interpolate(x1, t1, x0=x0_1)
        v_c1 = backbone(x_t1, t1, class_idx=labels)
        v_u1 = backbone(x_t1, t1, class_idx=null)
        # Reuse the library's flow-matching velocity loss (with the same
        # prediction/loss clamping used inside DeltaAlignmentLoss), summed over
        # the conditional + unconditional branches -- identical to the align
        # path's flow term for a single timestep.
        loss_flow = (
            flow_matching_velocity_loss(v_c1, target_1)
            + flow_matching_velocity_loss(v_u1, target_1)
        )
        total = args.lambda_flow * loss_flow
        return total, {"loss_flow": loss_flow.detach()}

    # A second timestep is required only for the cosine alignment term below.
    if getattr(args, "align_t_window", False):
        span = args.t_high - args.t_low
        t2 = args.t_low + span * torch.rand(B, device=device)
    else:
        t2 = torch.rand(B, device=device)

    # ------------------------------------------------------------------
    # FLOW PHASE of an align=True run (it < align_start, so the schedule has
    # set loss_fn.lambda_align == 0). Compute the SAME two-timestep flow loss
    # that loss_fn would, but skip the alignment term and its projector/feature
    # extraction. This is training-IDENTICAL -- with lambda_align == 0 the
    # alignment term contributes exactly zero gradient -- so we neither waste
    # compute on the (zero-weighted) projector nor log a misleading loss_align
    # during the first (1 - align_frac) of iterations. The final align_frac of
    # iterations sets lambda_align > 0 and falls through to the full path below.
    # ------------------------------------------------------------------
    if float(getattr(loss_fn, "lambda_align", 0.0) or 0.0) == 0.0:
        x_t1, target_1 = interpolant.interpolate(x1, t1, x0=x0_1)
        x_t2, target_2 = interpolant.interpolate(x1, t2, x0=x0_2)
        v_c1 = backbone(x_t1, t1, class_idx=labels)
        v_u1 = backbone(x_t1, t1, class_idx=null)
        v_c2 = backbone(x_t2, t2, class_idx=labels)
        v_u2 = backbone(x_t2, t2, class_idx=null)
        # Match DeltaAlignmentLoss.forward exactly: per-mode flow loss averaged
        # over the two timesteps, then summed over conditional + unconditional.
        loss_flow_c = (
            flow_matching_velocity_loss(v_c1, target_1)
            + flow_matching_velocity_loss(v_c2, target_2)
        ) / 2
        loss_flow_u = (
            flow_matching_velocity_loss(v_u1, target_1)
            + flow_matching_velocity_loss(v_u2, target_2)
        ) / 2
        loss_flow = loss_flow_c + loss_flow_u
        total = args.lambda_flow * loss_flow
        return total, {"loss_total": total.detach(), "loss_flow": loss_flow.detach()}

    x_t1, target_1 = interpolant.interpolate(x1, t1, x0=x0_1)
    x_t2, target_2 = interpolant.interpolate(x1, t2, x0=x0_2)

    v_c1, feats_c1 = backbone.forward_with_features(x_t1, t1, class_idx=labels)
    v_u1, feats_u1 = backbone.forward_with_features(x_t1, t1, class_idx=null)
    v_c2, feats_c2 = backbone.forward_with_features(x_t2, t2, class_idx=labels)
    v_u2, feats_u2 = backbone.forward_with_features(x_t2, t2, class_idx=null)

    return loss_fn(
        v_c1, v_u1, target_1,
        v_c2, v_u2, target_2,
        feats_u1, feats_c1, feats_u2, feats_c2,
    )


def _infinite(loader):
    """Yield batches endlessly, reshuffling each pass -- for iteration budgets."""
    while True:
        for batch in loader:
            yield batch


def _ema_path(out):
    """Sibling path for the EMA shadow weights: ``foo.pt`` -> ``foo.ema.pt``."""
    root, ext = os.path.splitext(out)
    return root + ".ema" + (ext or ".pt")


def _init_ema(backbone):
    """Snapshot the full state_dict (params + buffers) as EMA shadow tensors."""
    return {k: v.detach().clone() for k, v in backbone.state_dict().items()}


def _update_ema(ema_state, backbone, decay):
    """In-place EMA update: floating tensors decay toward the live weights,
    integer buffers are copied verbatim. Cheap enough to run every step."""
    for k, v in backbone.state_dict().items():
        e = ema_state[k]
        if v.dtype.is_floating_point:
            e.mul_(decay).add_(v.detach(), alpha=1.0 - decay)
        else:
            e.copy_(v)


def _save_checkpoint(backbone, out, ema_state=None):
    """Kill-safe checkpoint: always save the primary backbone weights, plus the
    EMA shadow to the sibling ``*.ema.pt`` when EMA is enabled. Centralised so
    every save site (per-iteration, per-epoch, final) behaves identically."""
    torch.save(backbone.state_dict(), out)
    if ema_state is not None:
        torch.save(ema_state, _ema_path(out))


def run_training(args):
    """Run guidance-aligned CFM pretraining. Returns ``(backbone, history)``
    where ``history`` is a list of per-logged-step dicts (for plotting)."""
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")

    dataset = build_dataset(args)
    print(f"dataset size: {len(dataset)}")
    if len(dataset) < args.batch_size:
        raise ValueError(
            f"dataset has {len(dataset)} images but --batch-size is "
            f"{args.batch_size}; lower --batch-size or raise --n-shot."
        )
    loader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True, drop_last=True)

    backbone = ConditionalUNetVelocityField(
        in_channels=1, cond_channels=1, num_classes=getattr(args, "num_classes", 0)
    ).to(device)

    # PURE-FLOW ABLATION: when alignment is disabled we skip the projector and
    # the alignment loss entirely (see _compute_batch_loss), so there is nothing
    # extra to build or optimise -- just the flow-matching backbone.
    align = getattr(args, "align", True)
    if align:
        feature_dims = backbone.feature_channels()
        projector = MultiScaleProjector(feature_dims, hidden_dim=64, out_dim=32).to(device)
        loss_fn = DeltaAlignmentLoss(
            projector, lambda_flow=args.lambda_flow, lambda_align=args.lambda_align
        )
        opt_params = list(backbone.parameters()) + list(projector.parameters())
    else:
        projector = None
        loss_fn = None
        opt_params = list(backbone.parameters())
        print("[pure-flow] alignment DISABLED: flow-matching loss only "
              "(no projector, single timestep, 2 fwd passes/step). "
              "This DIVERGES from CDPM-Align.")

    # BACKBONE OBJECTIVE (independent variable). "cfm" = flow-matching velocity
    # regression along a chosen probability PATH; "ddpm" = CDPM's DDPM
    # noise-prediction objective (paper reproduction / internal baseline). The
    # objective is captured ENTIRELY by the interpolant + target (the UNet, the
    # delta-alignment loss and the OT coupling are shared), so selecting "ddpm"
    # here is a clean single-variable swap for the iso-compute head-to-head.
    backbone_kind = str(getattr(args, "backbone", "cfm")).lower()
    if backbone_kind == "ddpm":
        interpolant = DDPMInterpolant()
        print("backbone=ddpm: DDPM noise-prediction objective "
              "(linear beta 1e-4->0.028, T=500; target=noise epsilon). "
              "The 'interpolant'/'sb_sigma' path knobs are IGNORED. For a "
              "faithful paper reproduction run with ot_coupling=False.")
    else:
        # PROBABILITY PATH (independent variable). "linear" = straight-line
        # rectified-flow path; "schrodinger" = Brownian-bridge path with tunable
        # diffusivity sb_sigma (deltaflow SchrodingerBridgeInterpolant). Paired
        # with OT coupling, a SMALL sb_sigma approximates the true entropic
        # Schrodinger bridge (unregularised OT is the small-sigma limit). The
        # coupling (independent vs OT) and the path (linear vs SB) are orthogonal
        # knobs.
        interp_name = str(getattr(args, "interpolant", "linear")).lower()
        if interp_name in ("schrodinger", "sb", "schrodinger_bridge"):
            sb_sigma = float(getattr(args, "sb_sigma", 0.1) or 0.0)
            interpolant = SchrodingerBridgeInterpolant(sigma=sb_sigma)
            print(f"interpolant: Schrodinger bridge (sigma={sb_sigma})")
        else:
            interpolant = LinearInterpolant()
            print("interpolant: linear (rectified-flow)")
    opt = torch.optim.Adam(opt_params, lr=args.lr)

    # OPTIONAL EMA of the backbone weights (disabled by default: ema_decay=0.0).
    # When enabled, a shadow copy tracks an exponential moving average of the
    # weights and is written to a sibling ``*.ema.pt`` file at every checkpoint.
    # The primary ``args.out`` save is UNCHANGED, so default behaviour and the
    # downstream probe (which loads args.out) are unaffected until you opt in and
    # point the sampler/probe at the EMA path. EMA typically yields smoother,
    # less noisy samples -- useful given the very small batch size here.
    ema_decay = float(getattr(args, "ema_decay", 0.0) or 0.0)
    use_ema = ema_decay > 0.0
    ema_state = _init_ema(backbone) if use_ema else None
    if use_ema:
        print(f"[ema] ENABLED (decay={ema_decay}); shadow -> {_ema_path(args.out)}")

    history = []
    step = 0
    # Wall-clock safety: bumping the budget on the larger 4-pass backbone can
    # exceed Kaggle's 9/12h limit. Save a checkpoint periodically so a hard kill
    # still leaves the latest usable backbone, and soft-stop before
    # ``max_seconds`` (a controlled compute budget -- match it across the DDPM vs
    # CFM runs for a fair claim). ``max_seconds=None`` disables the soft stop.
    t_start = time.time()
    max_seconds = getattr(args, "max_seconds", None)

    # --------------------------------------------------------------------
    # CDPM-faithful ITERATION-BUDGET mode (preferred). CDPM-Align measures
    # pretraining in *iterations*, not epochs: 50k total = 45k standard
    # diffusion + 5k alignment fine-tuning, where the alignment loss is only
    # active in the FINAL 10% of iterations (paper Sec. 3), at an effective
    # batch size of 16. We reproduce all three controls here:
    #   * ``max_iters``  -> total optimiser updates (CDPM: 50000)
    #   * ``grad_accum`` -> micro-batches per update; effective batch =
    #                       batch_size * grad_accum (set to reach CDPM's 16 on
    #                       a T4 that can only fit batch_size=2)
    #   * ``align_frac`` -> alignment active only in the final fraction of
    #                       iterations (CDPM: 0.1). Before that lambda_align=0,
    #                       so the first 90% is pure flow matching. This both
    #                       matches the paper's schedule AND avoids the
    #                       projector-collapse (loss_align -> 0) seen when the
    #                       alignment term is applied from step 0.
    # The flow-matching loss trains BOTH the conditional and unconditional
    # (null-token) velocity branches every step, so classifier-free guidance is
    # learned throughout regardless of the alignment schedule.
    # Set ``max_iters=None`` to fall back to the legacy epoch loop.
    # --------------------------------------------------------------------
    max_iters = getattr(args, "max_iters", None)
    grad_accum = max(1, int(getattr(args, "grad_accum", 1) or 1))

    if max_iters:
        max_iters = int(max_iters)
        align_frac = float(getattr(args, "align_frac", 0.1) or 0.0)
        align_start = int(round((1.0 - align_frac) * max_iters))
        ckpt_every = int(getattr(args, "ckpt_every", 500) or 500)
        eff_batch = args.batch_size * grad_accum
        base_align = args.lambda_align if align else 0.0
        print(f"iteration-budget: max_iters={max_iters} grad_accum={grad_accum} "
              f"eff_batch={eff_batch} align={align} lambda_align={base_align} "
              f"align_start={align_start} (final {align_frac:.0%}) "
              f"ckpt_every={ckpt_every}")
        data_iter = _infinite(loader)
        ot_coupling = bool(getattr(args, "ot_coupling", False))
        bs = args.batch_size
        # Mini-batch OT coupling via the deltaflow library (OTCoupling.sample_pair
        # draws x0 ~ N(0, I) and permutes it to minimise the batch's squared-L2
        # transport cost; Tong et al. 2023). Applied over the FULL effective batch
        # below, not per micro-batch. Kept only for the ot_coupling=True path so
        # the ot_coupling=False path stays byte-identical to the baseline.
        coupler = OTCoupling() if ot_coupling else None
        print(f"coupling: {'mini-batch OT over the full effective batch' if ot_coupling else 'independent (standard CFM)'}")
        for it in range(max_iters):
            # Alignment schedule: off for the first (1-align_frac), on after.
            # (Skipped entirely in the pure-flow ablation, where loss_fn is None.)
            if align:
                loss_fn.lambda_align = base_align if it >= align_start else 0.0

            opt.zero_grad()
            agg = {}
            # Draw all grad_accum micro-batches up front. With OT coupling this
            # lets the transport plan span the FULL effective batch
            # (batch_size * grad_accum), not a single micro-batch of size
            # batch_size (=2 here) where OT would be a near no-op. The noise is
            # OT-coupled once over the effective batch, then sliced back into
            # micro-batches for memory-safe gradient accumulation. When
            # ot_coupling is False, x0 stays None and each micro-batch draws
            # independent noise internally -- byte-identical to the baseline.
            micro = [next(data_iter) for _ in range(grad_accum)]
            x0_1_full = x0_2_full = None
            if ot_coupling:
                x1_full = torch.cat([mb[0] for mb in micro], dim=0).to(device)
                # Independent OT couplings for the two alignment timesteps, so
                # the ONLY change vs the independent-coupling baseline is the
                # coupling itself (each timestep still gets its own noise draw).
                x0_1_full, _ = coupler.sample_pair(x1_full)
                x0_2_full, _ = coupler.sample_pair(x1_full)
            for j, (x1, labels) in enumerate(micro):
                if ot_coupling:
                    sl = slice(j * bs, j * bs + x1.shape[0])
                    x0_1, x0_2 = x0_1_full[sl], x0_2_full[sl]
                else:
                    x0_1 = x0_2 = None
                total, loss_dict = _compute_batch_loss(
                    backbone, loss_fn, interpolant, x1, labels, args, device,
                    x0_1=x0_1, x0_2=x0_2,
                )
                (total / grad_accum).backward()
                for k, v in loss_dict.items():
                    agg[k] = agg.get(k, 0.0) + float(v.item()) / grad_accum
            opt.step()
            if use_ema:
                _update_ema(ema_state, backbone, ema_decay)
            step += 1

            if it % 10 == 0:
                phase = "align" if (align and it >= align_start) else "flow"
                record = {"step": it, "phase": phase,
                          "lambda_align": (loss_fn.lambda_align if align else 0.0),
                          **agg}
                history.append(record)
                print(f"iter={it:6d} [{phase}] " + " ".join(
                    f"{k}={agg[k]:.4f}" for k in
                    ("loss_total", "loss_flow", "loss_align") if k in agg
                ))

            elapsed = time.time() - t_start
            if (it + 1) % ckpt_every == 0:
                _save_checkpoint(backbone, args.out, ema_state if use_ema else None)
                print(f"[iter {it+1}] elapsed={elapsed/60:.1f} min | ckpt -> {args.out}")
            if max_seconds is not None and elapsed >= max_seconds:
                print(f"[soft-stop] reached max_seconds={max_seconds:.0f}s at "
                      f"iter {it}; stopping with {it+1} updates trained.")
                break

        _save_checkpoint(backbone, args.out, ema_state if use_ema else None)
        if use_ema:
            print(f"[ema] saved EMA weights -> {_ema_path(args.out)}")
        print(f"saved: {args.out}")
        return backbone, history

    # --------------------------------------------------------------------
    # Legacy EPOCH-BUDGET mode (kept for CLI back-compat / quick smoke tests).
    # --------------------------------------------------------------------
    stop = False
    for epoch in range(args.epochs):
        for x1, labels in loader:
            total, loss_dict = _compute_batch_loss(
                backbone, loss_fn, interpolant, x1, labels, args, device
            )

            opt.zero_grad()
            total.backward()
            opt.step()
            if use_ema:
                _update_ema(ema_state, backbone, ema_decay)

            if step % 10 == 0:
                record = {"epoch": epoch, "step": step,
                          **{k: float(v.item()) for k, v in loss_dict.items()}}
                history.append(record)
                print(f"epoch={epoch:3d} step={step:5d} " + " ".join(
                    f"{k}={v:.4f}" for k, v in record.items()
                    if k not in ("epoch", "step")
                ))
            step += 1

        # End-of-epoch checkpoint (crash/kill-safe) + soft time-budget stop.
        _save_checkpoint(backbone, args.out, ema_state if use_ema else None)
        elapsed = time.time() - t_start
        print(f"[epoch {epoch} done] elapsed={elapsed/60:.1f} min | ckpt -> {args.out}")
        if max_seconds is not None and elapsed >= max_seconds:
            print(f"[soft-stop] reached max_seconds={max_seconds:.0f}s at epoch "
                  f"{epoch}; stopping with {step} steps trained.")
            stop = True
        if stop:
            break

    _save_checkpoint(backbone, args.out, ema_state if use_ema else None)
    if use_ema:
        print(f"[ema] saved EMA weights -> {_ema_path(args.out)}")
    print(f"saved: {args.out}")
    return backbone, history


def build_arg_parser():
    p = argparse.ArgumentParser()
    p.add_argument(
        "--dataset", choices=["combined", "chest", "isbi2015"],
        default="combined",
    )
    p.add_argument("--root", required=True)
    p.add_argument("--chest-root", default=None,
                   help="Shenzhen chest image dir (dataset index 1) for --dataset combined")
    p.add_argument("--isbi-root", default=None,
                   help="ISBI2015 cephalometric image dir (dataset index 2) for --dataset combined")
    p.add_argument("--dha-root", default=None,
                   help="Digital Hand Atlas image dir (dataset index 3) for --dataset combined")
    p.add_argument("--num-classes", type=int, default=0,
                   help="dataset-index conditioning table size (0=off; index 0 is the null token)")
    p.add_argument("--pretrain-limit", type=int, default=None,
                   help="cap images per source for --dataset combined (None=use all)")
    p.add_argument("--no-exclude-test", dest="exclude_test", action="store_false",
                   help="do NOT hold downstream test images out of the combined "
                        "pretraining corpus (default: exclude, to avoid leakage)")
    p.set_defaults(exclude_test=True)
    p.add_argument("--chest-pp-root", default=None,
                   help="crop+pad Shenzhen image dir used downstream (to reproduce "
                        "and hold out its seeded test split during pretraining)")
    p.add_argument("--chest-landmarks-dir", default=None,
                   help="Shenzhen landmark .npy dir (to identify chest test images)")
    p.add_argument("--test-frac", type=float, default=0.2,
                   help="downstream chest test fraction (must match the probe's, "
                        "so pretraining excludes exactly the probe test images)")
    p.add_argument("--landmarks-dir", default=None, help="required for --dataset chest")
    p.add_argument("--landmark-ref-size", type=int, default=1024,
                   help="resolution the --dataset chest landmarks were annotated at")
    p.add_argument("--split", default="train")
    p.add_argument("--image-size", type=int, default=128)
    p.add_argument("--batch-size", type=int, default=8)
    p.add_argument("--epochs", type=int, default=30,
                   help="epoch budget (legacy mode; ignored when --max-iters is set)")
    p.add_argument("--max-iters", type=int, default=None,
                   help="CDPM-faithful iteration budget (optimiser updates). "
                        "CDPM-Align uses 50000. None => legacy epoch loop.")
    p.add_argument("--grad-accum", type=int, default=1,
                   help="micro-batches accumulated per update; effective batch = "
                        "batch_size * grad_accum (set to reach CDPM's 16).")
    p.add_argument("--ot-coupling", dest="ot_coupling", action="store_true",
                   help="mini-batch optimal-transport coupling: re-order the "
                        "noise within each EFFECTIVE batch (batch_size * "
                        "grad_accum) to minimise squared-L2 transport cost "
                        "(Tong et al. 2023) for straighter flow paths.")
    p.add_argument("--no-ot-coupling", dest="ot_coupling", action="store_false",
                   help="independent noise-data coupling (standard CFM; default).")
    p.set_defaults(ot_coupling=False)
    p.add_argument("--interpolant", choices=["linear", "schrodinger"],
                   default="linear",
                   help="probability path: 'linear' (rectified flow) or "
                        "'schrodinger' (Brownian-bridge path with --sb-sigma). "
                        "Ignored when --backbone ddpm.")
    p.add_argument("--backbone", choices=["cfm", "ddpm"], default="cfm",
                   help="generative objective (the independent variable): 'cfm' "
                        "= flow-matching velocity regression (this work); 'ddpm' "
                        "= CDPM's DDPM noise-prediction objective (paper repro / "
                        "iso-compute baseline). 'ddpm' overrides --interpolant.")
    p.add_argument("--sb-sigma", type=float, default=0.1,
                   help="Schrodinger-bridge diffusivity (only used when "
                        "--interpolant schrodinger). 0 recovers the linear path.")
    p.add_argument("--align-frac", type=float, default=0.1,
                   help="alignment loss active only in the final fraction of "
                        "iterations (CDPM: 0.1). 0 disables alignment entirely.")
    p.add_argument("--no-align", dest="align", action="store_false",
                   help="PURE-FLOW ABLATION: disable the multi-scale guidance "
                        "alignment loss entirely (single-timestep flow matching, "
                        "no projector; ~2x faster). Diverges from CDPM-Align.")
    p.set_defaults(align=True)
    p.add_argument("--ckpt-every", type=int, default=500,
                   help="save a kill-safe checkpoint every N iterations "
                        "(iteration-budget mode only).")
    p.add_argument("--n-shot", type=int, default=None)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--lambda-flow", type=float, default=1.0)
    p.add_argument("--lambda-align", type=float, default=5.0)
    p.add_argument("--t-low", type=float, default=0.25,
                   help="lower bound for the mid-range timestep sampling window")
    p.add_argument("--t-high", type=float, default=0.75,
                   help="upper bound for the mid-range timestep sampling window")
    p.add_argument("--out", default="cfm_delta_align_backbone.pt")
    p.add_argument("--max-seconds", type=float, default=None,
                   help="soft wall-clock budget; stop after the epoch that crosses it "
                        "(checkpoint is saved every epoch regardless). None disables.")
    p.add_argument("--ema-decay", type=float, default=0.0,
                   help="EMA decay for the backbone weights (e.g. 0.999). 0.0 "
                        "disables EMA (default). When >0, an EMA shadow is saved "
                        "to a sibling *.ema.pt at every checkpoint.")
    p.add_argument("--seed", type=int, default=0)
    return p


def main():
    args = build_arg_parser().parse_args()
    run_training(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prepare_shenzhen_landmarks.py
"""
Align ngaggion lung landmarks with the Shenzhen chest X-rays so the downstream
probe can be trained/evaluated on *real* anatomy instead of Sobel
pseudo-landmarks.

The ngaggion annotations live in a 1024x1024 frame produced by crop-to-content
-> pad-to-square -> resize. We replicate that exact crop+pad on each raw
Shenzhen image; after our dataset resizes to image_size, a landmark maps by the
simple scale image_size / 1024 (passed as landmark_ref_size=1024).

Outputs
  <out_images>/<stem>.png      cropped+padded images (still original resolution)
  <out_landmarks>/<stem>.npy   combined RL+LL landmarks, (94, 2), in 1024 space
"""

import argparse
from pathlib import Path

import numpy as np

try:
    import cv2
except ImportError:  # pragma: no cover
    cv2 = None


def preprocess_to_square(img: np.ndarray) -> np.ndarray:
    """Crop to the content bounding box and pad to a square (ngaggion style)."""
    gray = (255 * (img > 1)).astype("uint8")
    coords = cv2.findNonZero(gray)
    x, y, w, h = cv2.boundingRect(coords)
    crop = img[y : y + h, x : x + w]
    sh = crop.shape
    if sh[0] < sh[1]:
        pad = sh[1] - sh[0]
        pad_y = [pad // 2, pad - pad // 2]
        pad_x = [0, 0]
    elif sh[1] < sh[0]:
        pad = sh[0] - sh[1]
        pad_x = [pad // 2, pad - pad // 2]
        pad_y = [0, 0]
    else:
        pad_x = pad_y = [0, 0]
    return np.pad(crop, [pad_y, pad_x])


def prepare(images_dir, landmarks_root, out_images, out_landmarks) -> int:
    """Emit cropped+padded images and combined RL+LL landmark files."""
    if cv2 is None:
        raise ImportError("opencv-python is required")

    images_dir = Path(images_dir)
    landmarks_root = Path(landmarks_root)
    out_images = Path(out_images)
    out_landmarks = Path(out_landmarks)
    out_images.mkdir(parents=True, exist_ok=True)
    out_landmarks.mkdir(parents=True, exist_ok=True)

    rl_dir = landmarks_root / "RL"
    ll_dir = landmarks_root / "LL"
    if not rl_dir.is_dir() or not ll_dir.is_dir():
        raise FileNotFoundError(f"expected 'RL' and 'LL' subfolders under {landmarks_root}")

    count = 0
    for rl_path in sorted(rl_dir.glob("*.npy")):
        stem = rl_path.stem
        ll_path = ll_dir / f"{stem}.npy"
        img_path = images_dir / f"{stem}.png"
        if not ll_path.exists() or not img_path.exists():
            continue

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        cv2.imwrite(str(out_images / f"{stem}.png"), preprocess_to_square(img))

        rl = np.load(rl_path).reshape(-1, 2)
        ll = np.load(ll_path).reshape(-1, 2)
        combined = np.concatenate([rl, ll], axis=0).astype("float32")
        np.save(out_landmarks / f"{stem}.npy", combined)
        count += 1

    return count


def build_arg_parser():
    p = argparse.ArgumentParser()
    p.add_argument("--images-dir", required=True, help="raw Shenzhen PNG folder")
    p.add_argument("--landmarks-root", required=True,
                   help="ngaggion 'landmarks' folder (contains RL/ and LL/)")
    p.add_argument("--out-images", default="shenzhen_pp")
    p.add_argument("--out-landmarks", default="shenzhen_landmarks")
    return p


def main():
    args = build_arg_parser().parse_args()
    n = prepare(args.images_dir, args.landmarks_root, args.out_images, args.out_landmarks)
    print(f"wrote {n} image/landmark pairs")
    print(f"  images    -> {args.out_images}")
    print(f"  landmarks -> {args.out_landmarks} (94 pts each, 1024 frame)")


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, sys
from pathlib import Path
import numpy as np

# 1) Clone the landmark annotations (small; images are NOT needed from here).
LM_REPO = Path("Chest-xray-landmark-dataset")
if not LM_REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/ngaggion/Chest-xray-landmark-dataset.git", str(LM_REPO)],
        check=True,
    )

# 2) Locate the folder that actually contains RL/ and LL/ subfolders.
LM_ROOT = None
for rl in LM_REPO.rglob("RL"):
    if rl.is_dir() and (rl.parent / "LL").is_dir():
        LM_ROOT = rl.parent
        break
assert LM_ROOT is not None, "could not find RL/ and LL/ landmark folders in the clone"
print("landmark root:", LM_ROOT)

# 3) Build aligned images + combined (94,2) landmarks. ROOT is the raw Shenzhen
#    PNG folder discovered in section 2.
from prepare_shenzhen_landmarks import prepare  # noqa: E402

PP_ROOT = "shenzhen_pp"          # cropped+padded images (dataset root for 'chest')
LM_DIR = "shenzhen_landmarks"    # per-image combined landmarks, 1024 frame
n_pairs = prepare(ROOT, LM_ROOT, PP_ROOT, LM_DIR)
print(f"matched {n_pairs} Shenzhen images with real lung landmarks")



In [ ]:
import os, sys
from types import SimpleNamespace

sys.path.insert(0, os.getcwd())  # so `cfm_pretrain` (just written) is importable
from cfm_pretrain import run_training

set_all_seeds(E.seed)  # re-pin now that torch is installed

# All CONTROLLED variables come from EXPERIMENT (single source of truth); only
# `backbone` (the independent variable) distinguishes this from a DDPM run.
args = SimpleNamespace(
    backbone=getattr(E, "backbone", "cfm"),
    dataset=E.dataset, root=ROOT, chest_root=ROOT, isbi_root=ISBI_ROOT,
    dha_root=DHA_ROOT,
    num_classes=E.num_classes, pretrain_limit=E.pretrain_limit,
    pretrain_targets=getattr(E, "pretrain_targets", None),
    # No-leakage: hold every downstream TEST image out of the pooled
    # pretraining corpus (CDPM-Align: 'corpus, excluding the test set').
    exclude_test=True, chest_pp_root=PP_ROOT, chest_landmarks_dir=LM_DIR,
    test_frac=E.test_frac,
    landmarks_dir=None, landmark_ref_size=1024,
    split=E.split, image_size=E.image_size, batch_size=E.batch_size,
    epochs=E.epochs_pretrain, n_shot=E.n_shot,
    max_iters=getattr(E, "pretrain_iters", None),
    grad_accum=getattr(E, "grad_accum", 1),
    ema_decay=getattr(E, "ema_decay", 0.0),
    align_frac=getattr(E, "align_frac", 0.1),
    align=getattr(E, "align", True),
    ot_coupling=getattr(E, "ot_coupling", False),
    interpolant=getattr(E, "interpolant", "linear"),
    sb_sigma=getattr(E, "sb_sigma", 0.1),
    ckpt_every=getattr(E, "ckpt_every", 500),
    lr=E.lr_pretrain, lambda_flow=E.lambda_flow, lambda_align=E.lambda_align,
    t_low=E.t_low, t_high=E.t_high,
    out=os.path.join(OUT_DIR, "cfm_delta_align_backbone.pt"),
    max_seconds=getattr(E, "max_pretrain_seconds", None), seed=E.seed,
)
print("pretraining config:", vars(args))
backbone, history = run_training(args)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --------------------------------------------------------------------------
# Readable pretraining loss curve. Raw per-step values are very noisy, and the
# large initial spike compresses a linear axis so later dynamics look flat. So:
#   * plot the raw trace faint + an EMA-smoothed bold line on top,
#   * put the flow/total loss on a LOG y-axis (reveals post-spike decay),
#   * shade the final alignment phase (where loss_align actually contributes),
#   * degrade gracefully to a single axis for the pure-flow (align=False) run,
#     where only loss_flow is logged.
# --------------------------------------------------------------------------
def _ema(y, alpha=0.15):
    y = np.asarray(y, dtype=float)
    if len(y) == 0:
        return y
    out = np.empty_like(y); acc = y[0]
    for i, v in enumerate(y):
        acc = alpha * v + (1.0 - alpha) * acc
        out[i] = acc
    return out

steps = np.array([h["step"] for h in history])
keys = set().union(*[h.keys() for h in history]) if history else set()
has_align = "loss_align" in keys
has_total = "loss_total" in keys

flow = np.array([h["loss_flow"] for h in history])

fig, ax1 = plt.subplots(figsize=(9, 5))

# Shade the alignment phase (iteration-budget runs tag each record with "phase").
align_start = None
_align_steps = [h["step"] for h in history if h.get("phase") == "align"]
if _align_steps:
    align_start = min(_align_steps)
    ax1.axvspan(align_start, steps.max(), color="0.90", zorder=0)
    ax1.axvline(align_start, color="0.55", ls="--", lw=1, zorder=1)
    ax1.text(align_start, 0.5, "alignment on (final %d%%)  "
             % round(100 * (steps.max() - align_start) / max(1, steps.max())),
             transform=ax1.get_xaxis_transform(), va="center", ha="right",
             rotation=90, fontsize=8, color="0.35")

# Left axis: flow (+ total) loss, log-scaled, raw faint + EMA bold.
ax1.plot(steps, flow, color="C1", lw=0.7, alpha=0.25)
ax1.plot(steps, _ema(flow), color="C1", lw=2.2, label="loss_flow (EMA)")
if has_total:
    tot = np.array([h["loss_total"] for h in history])
    ax1.plot(steps, tot, color="C0", lw=0.7, alpha=0.25)
    ax1.plot(steps, _ema(tot), color="C0", lw=2.2, ls="--", label="loss_total (EMA)")
ax1.set_yscale("log")
ax1.set_xlabel("training step")
ax1.set_ylabel("flow / total loss  (log scale)")
ax1.grid(alpha=0.3, which="both")

handles, labels = ax1.get_legend_handles_labels()

# Right axis: alignment loss (own scale; typically << flow loss). Only the
# final align_frac of iterations logs loss_align (the flow phase omits it, since
# lambda_align == 0 there), so plot it over just those steps -- which coincides
# with the shaded alignment region above.
if has_align:
    al_steps = np.array([h["step"] for h in history if "loss_align" in h])
    al = np.array([h["loss_align"] for h in history if "loss_align" in h])
    ax2 = ax1.twinx()
    ax2.plot(al_steps, al, color="C2", lw=0.6, alpha=0.20)
    ax2.plot(al_steps, _ema(al), color="C2", lw=2.2, label="loss_align (EMA)")
    ax2.set_ylabel("align loss  (1 - cos)  [goal -> 0]", color="C2")
    ax2.tick_params(axis="y", labelcolor="C2")
    h2, l2 = ax2.get_legend_handles_labels()
    handles += h2; labels += l2

ax1.legend(handles, labels, loc="upper right", framealpha=0.9)
title = ("CFM pretraining (pure-flow, EMA-smoothed)" if not has_align
         else "CFM + delta-alignment pretraining (EMA-smoothed)")
plt.title(title)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "loss_curves.png"), dpi=120)
plt.show()
print("Saved backbone ->", args.out)


## 5. Downstream probe: few-shot landmark detection (module + utilities)

The pretraining above is only useful if its representation *transfers*. The **next cell** writes the probe module (`probe_landmark_detection.py`): a small multi-scale heatmap head on the backbone's **unconditional** features, trained with CDPM's spatial-softmax **NLL** loss (full fine-tune, AdamW, **early stopping on a held-out validation split -- never the test set**) and scored with **MRE / SDR@2mm / P95** on the fixed test set. Every evaluation runs **twice** -- *pretrained* vs *random-init* -- to isolate whether CFM pretraining actually helps.

The cell after it imports `run_probe` and a small `best()` helper, which now selects the epoch by **val MRE** so the reported test metrics are not chosen by peeking at the test set. The real evaluations then follow:
- **§6** -- real anatomical lung landmarks (the honest transfer signal), and
- **§6.5** -- ISBI2015 cephalometric landmarks in **millimetres**, the head-to-head vs CDPM-Align.

> The earlier pseudo-landmark (`chest-auto`) sanity run has been **removed**: its target was derived from the same heuristic the backbone was pretrained to condition on, so a low MRE there was partly *circular*. §6 replaces it with independent, human-annotated anatomy.

In [ ]:
%%writefile probe_landmark_detection.py
"""Downstream few-shot landmark-detection probe for a pretrained CFM +
delta-alignment backbone. Importable: call run_probe(args).

Downstream protocol is aligned to CDPM-Align (the controlled variable in this
study, so the backbone is the ONLY difference vs the paper):
  * spatial-softmax + NLL heatmap loss (CDPM CustomNLLLoss), NOT background-
    dominated heatmap MSE;
  * full fine-tune of the backbone (freeze_backbone=False) by default;
  * AdamW (weight_decay) + early stopping on test MRE (patience).
An unnormalised-Gaussian MSE head is still selectable (loss="mse") as a
diagnostic.
"""

import copy
import random
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from deltaflow.models.conditional_unet import ConditionalUNetVelocityField

try:
    from cfm_pretrain import build_dataset
except ImportError:
    from pretrain_cfm_delta_align import build_dataset


def landmarks_to_target_heatmaps(landmarks, image_size, sigma=3.0):
    """(K, 2) landmarks -> (K, H, W) per-landmark Gaussian targets (peak 1)."""
    device = landmarks.device
    ys = torch.arange(image_size, device=device).view(1, -1, 1)
    xs = torch.arange(image_size, device=device).view(1, 1, -1)
    lx = landmarks[:, 0].view(-1, 1, 1)
    ly = landmarks[:, 1].view(-1, 1, 1)
    return torch.exp(-((xs - lx) ** 2 + (ys - ly) ** 2) / (2 * sigma**2))


def landmarks_to_prob_target(landmarks, image_size, sigma=3.0):
    """(K, 2) landmarks -> (K, H, W) per-landmark PROBABILITY targets that sum
    to 1 over the spatial map (a valid distribution for NLL / cross-entropy).

    sigma > 0 gives a normalised Gaussian (soft label); sigma <= 0 gives a
    one-hot target at the rounded pixel (CDPM cdpm_align uses sigma=0)."""
    K = landmarks.shape[0]
    device = landmarks.device
    if sigma and sigma > 0:
        hm = landmarks_to_target_heatmaps(landmarks, image_size, sigma)
        denom = hm.sum(dim=(-1, -2), keepdim=True).clamp_min(1e-8)
        return hm / denom
    hm = torch.zeros(K, image_size, image_size, device=device)
    xy = landmarks.round().long().clamp_(0, image_size - 1)
    hm[torch.arange(K, device=device), xy[:, 1], xy[:, 0]] = 1.0
    return hm


def two_d_softmax(logits):
    """(B, K, H, W) -> softmax over the H*W spatial dimension per (B, K)."""
    B, K, H, W = logits.shape
    return F.softmax(logits.view(B, K, -1), dim=-1).view(B, K, H, W)


def heatmap_nll_loss(logits, target_prob, eps=1e-10):
    """CDPM-style spatial cross-entropy: -sum_{H,W} target * log(softmax(logits)),
    averaged over batch and landmarks. Immune to background dominance because the
    prediction is normalised over the map."""
    pred = two_d_softmax(logits)
    nll = -(target_prob * torch.log(pred + eps)).sum(dim=(-1, -2))
    return nll.mean()


def heatmaps_to_coords(heatmaps, subpixel=True):
    """(B, K, H, W) -> (B, K, 2) peak coords as (x, y).

    CDPM-Align's ``get_hottest_point`` refines the integer argmax with a
    quadratic (parabolic) sub-pixel fit along each axis, rather than returning
    the raw argmax. At 256px one pixel is ~0.75-0.94 mm on ISBI2015, so the
    ~0.3 px average quantisation error of a plain argmax is a non-trivial slice
    of the mm-scale MRE we are trying to beat. ``subpixel=True`` (default)
    matches CDPM; set it False for the old integer-argmax behaviour.

    Parabolic offset along an axis from the peak value ``c0`` and its two
    neighbours ``cm1``/``cp1``:  ``delta = 0.5*(cm1 - cp1)/(cm1 - 2*c0 + cp1)``,
    clamped to [-0.5, 0.5]; the denominator is guarded against ~0 curvature.
    """
    B, K, H, W = heatmaps.shape
    idx = heatmaps.view(B, K, -1).argmax(dim=-1)
    ys = idx // W
    xs = idx % W
    xs_f = xs.float()
    ys_f = ys.float()
    if subpixel:
        bi = torch.arange(B, device=heatmaps.device).view(B, 1)
        ki = torch.arange(K, device=heatmaps.device).view(1, K)

        def _off(cm1, c0, cp1):
            denom = cm1 - 2.0 * c0 + cp1
            off = 0.5 * (cm1 - cp1) / denom
            off = torch.where(denom.abs() < 1e-6, torch.zeros_like(off), off)
            return off.clamp(-0.5, 0.5)

        xm1 = (xs - 1).clamp(0, W - 1)
        xp1 = (xs + 1).clamp(0, W - 1)
        ym1 = (ys - 1).clamp(0, H - 1)
        yp1 = (ys + 1).clamp(0, H - 1)
        c0 = heatmaps[bi, ki, ys, xs]
        xs_f = xs_f + _off(heatmaps[bi, ki, ys, xm1], c0, heatmaps[bi, ki, ys, xp1])
        ys_f = ys_f + _off(heatmaps[bi, ki, ym1, xs], c0, heatmaps[bi, ki, yp1, xs])
    return torch.stack([xs_f, ys_f], dim=-1)


class MultiScaleHeatmapHead(nn.Module):
    LEVELS = ("enc_1_4", "dec_1_8", "bottleneck")

    def __init__(self, num_landmarks, feature_dims, image_size):
        super().__init__()
        self.image_size = image_size
        in_ch = sum(feature_dims[k] for k in self.LEVELS)
        self.fuse = nn.Sequential(
            nn.Conv2d(in_ch, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.SiLU(),
            nn.Conv2d(128, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.SiLU(),
        )
        self.out = nn.Conv2d(64, num_landmarks, 1)

    def forward(self, feats):
        size = (self.image_size, self.image_size)
        parts = [
            F.interpolate(feats[k], size=size, mode="bilinear", align_corners=False)
            for k in self.LEVELS
        ]
        return self.out(self.fuse(torch.cat(parts, dim=1)))


class LandmarkProbe(nn.Module):
    def __init__(self, backbone, num_landmarks, feature_dims, image_size,
                 t_value=1.0, freeze=True):
        super().__init__()
        self.backbone = backbone
        self.head = MultiScaleHeatmapHead(num_landmarks, feature_dims, image_size)
        self.t_value = t_value
        self.freeze = freeze
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad_(False)

    def forward(self, x):
        B = x.shape[0]
        t = torch.full((B,), self.t_value, device=x.device, dtype=x.dtype)
        if self.freeze:
            self.backbone.eval()
            with torch.no_grad():
                _, feats = self.backbone.forward_with_features(x, t, cond=None)
            feats = {k: v.detach() for k, v in feats.items()}
        else:
            _, feats = self.backbone.forward_with_features(x, t, cond=None)
        return self.head(feats)


def compute_metrics(pred, gt, mm_per_pixel, thresholds=(2.0, 2.5, 3.0, 4.0)):
    mm = torch.as_tensor(mm_per_pixel, dtype=torch.float32)
    err = torch.linalg.norm((pred - gt) * mm, dim=-1).flatten()
    metrics = {
        "MRE": err.mean().item(),
        "std": err.std(unbiased=False).item(),
        "P95": torch.quantile(err, 0.95).item(),
    }
    for z in thresholds:
        metrics[f"SDR@{z}mm"] = (err < z).float().mean().item() * 100.0
    return metrics


@torch.no_grad()
def evaluate(probe, loader, device, mm_per_pixel):
    probe.eval()
    preds, gts = [], []
    for x, lm in loader:
        x = x.to(device)
        preds.append(heatmaps_to_coords(probe(x)).cpu())
        gts.append(lm.float())
    return compute_metrics(torch.cat(preds), torch.cat(gts), mm_per_pixel)


def _split_indices(n, test_frac, n_shot, seed, val_frac=0.15):
    """Deterministic seeded split -> (train_idx, val_idx, test_idx).

    The permutation's first ``round(n*test_frac)`` entries are the fixed held-out
    TEST set (never touched during training/selection). The remaining pool yields
    the few-shot TRAIN set (``n_shot`` images) and a disjoint VAL set used for
    early stopping / model selection -- selecting on VAL rather than TEST avoids
    the optimistic bias of peeking at the evaluation set. ``val_frac`` is a
    fraction of ``n`` (clamped to what the pool can spare after the train shots).
    """
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n, generator=g).tolist()
    test_size = max(1, int(round(n * test_frac)))
    test_idx = perm[:test_size]
    pool = perm[test_size:]
    train_idx = pool[:n_shot] if n_shot else pool
    if not train_idx:
        raise ValueError(f"empty train split (dataset size {n})")
    # VAL comes from the pool AFTER the few-shot train images (disjoint from both
    # train and test). Falls back to empty if the pool has nothing left to spare.
    val_start = len(train_idx)
    val_size = min(max(1, int(round(n * val_frac))), max(0, len(pool) - val_start))
    val_idx = pool[val_start:val_start + val_size]
    return train_idx, val_idx, test_idx


def build_eval_split(args):
    """Rebuild the exact (deterministic) train/val/test split used by run_probe.

    Returns (train_ds, val_ds, test_ds, mm_per_pixel, lm0). Because the split is
    fully seeded, calling this again reproduces the identical held-out test set --
    handy for qualitatively visualising a trained probe on its own test images.
    ``val_ds`` is used for early stopping / model selection so we never select on
    the test set. It may be an empty Subset if the pool has no spare images.
    """
    def _ds(phase, n_shot):
        return SimpleNamespace(
            dataset=args.dataset, root=args.root, landmarks_dir=args.landmarks_dir,
            landmark_ref_size=args.landmark_ref_size, split=args.split,
            image_size=args.image_size, n_shot=n_shot, phase=phase,
        )

    if args.dataset == "isbi2015":
        # CDPM-Align faithful protocol: sorted-filename split -- few-shot pool is
        # train=[:130], the fixed val=[130:150] (20 imgs, for model selection),
        # and the fixed test=[150:400] (250 imgs); mm is per-axis from the dataset
        # (NOT a single scalar).
        full_train = build_dataset(_ds("train", None))
        val_ds = build_dataset(_ds("val", None))
        test_ds = build_dataset(_ds("test", None))
        n = min(args.n_shot, len(full_train)) if args.n_shot else len(full_train)
        g = torch.Generator().manual_seed(args.seed)
        shot_idx = torch.randperm(len(full_train), generator=g).tolist()[:n]
        train_ds = Subset(full_train, shot_idx)
        mm_per_pixel = full_train.mm_per_pixel
        _, lm0 = full_train[shot_idx[0]]
        print(f"ISBI2015 CDPM split: train(n-shot)={len(train_ds)} "
              f"val={len(val_ds)} test={len(test_ds)} "
              f"mm/px={[round(m,4) for m in mm_per_pixel]}")
    else:
        dataset = build_dataset(_ds("all", None))
        train_idx, val_idx, test_idx = _split_indices(
            len(dataset), args.test_frac, args.n_shot, args.seed
        )
        train_ds = Subset(dataset, train_idx)
        val_ds = Subset(dataset, val_idx)
        test_ds = Subset(dataset, test_idx)
        mm_per_pixel = args.mm_per_pixel
        _, lm0 = dataset[train_idx[0]]
        print(f"dataset={len(dataset)}  train(n-shot)={len(train_ds)}  "
              f"val={len(val_ds)}  test={len(test_ds)}")
    return train_ds, val_ds, test_ds, mm_per_pixel, lm0


def run_probe(args):
    """Train the probe and return (probe, history) with per-epoch metrics.

    Recognised (optional) CDPM-protocol knobs, read via getattr with defaults:
      loss          : "nll" (default) | "mse"
      weight_decay  : AdamW weight decay (default 1e-4)
      patience      : early-stopping patience on VAL MRE, 0 disables (default 15)

    Model selection (early stopping + best-checkpoint restore) is done on the
    held-out VAL split, NOT the test set, so the reported test metrics are not
    optimistically biased by peeking at the evaluation set. When no val images are
    available (empty val split) it transparently falls back to selecting on test.
    """
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")

    loss_type = getattr(args, "loss", "nll")
    weight_decay = getattr(args, "weight_decay", 1e-4)
    patience = getattr(args, "patience", 15)

    train_ds, val_ds, test_ds, mm_per_pixel, lm0 = build_eval_split(args)

    K = lm0.shape[0]
    print(f"num landmarks: {K}")

    backbone = ConditionalUNetVelocityField(
        in_channels=1, cond_channels=1, num_classes=getattr(args, "num_classes", 0)
    )
    if args.backbone_ckpt:
        backbone.load_state_dict(torch.load(args.backbone_ckpt, map_location="cpu"))
        print(f"loaded pretrained backbone: {args.backbone_ckpt}")
    else:
        print("random-init backbone (from-scratch baseline)")
    backbone.to(device)

    feature_dims = backbone.feature_channels()
    probe = LandmarkProbe(
        backbone, K, feature_dims, args.image_size,
        t_value=args.t_value, freeze=args.freeze_backbone,
    ).to(device)
    trainable = [p for p in probe.parameters() if p.requires_grad]
    mode = "linear-probe (frozen)" if args.freeze_backbone else "fine-tune (full)"
    print(f"mode: {mode} | loss: {loss_type} | wd: {weight_decay} | patience: {patience}")
    opt = torch.optim.AdamW(trainable, lr=args.lr, weight_decay=weight_decay)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size)
    use_val = len(val_ds) > 0
    val_loader = DataLoader(val_ds, batch_size=args.batch_size) if use_val else None
    sel_name = "val" if use_val else "test"
    if not use_val:
        print("[warn] empty val split -> selecting on TEST (biased); "
              "increase dataset/pool size for an unbiased val split.")

    history = []
    best_sel = float("inf")
    best_state = None
    bad_epochs = 0
    for epoch in range(args.epochs):
        probe.train()
        if args.freeze_backbone:
            probe.backbone.eval()
        ep_loss, nb = 0.0, 0
        for x, lm in train_loader:
            x = x.to(device)
            lm = lm.to(device).float()
            if loss_type == "mse":
                target = torch.stack([
                    landmarks_to_target_heatmaps(lm[i], args.image_size, args.sigma)
                    for i in range(x.shape[0])
                ])
                loss = F.mse_loss(probe(x), target)
            else:
                target = torch.stack([
                    landmarks_to_prob_target(lm[i], args.image_size, args.sigma)
                    for i in range(x.shape[0])
                ])
                loss = heatmap_nll_loss(probe(x), target)
            opt.zero_grad()
            loss.backward()
            opt.step()
            ep_loss += loss.item()
            nb += 1

        # Report metrics on TEST; select (early-stop) on VAL so we never peek at
        # the test set. ``val_MRE`` is stored alongside the test metrics so the
        # caller can pick the val-selected epoch without touching test.
        metrics = evaluate(probe, test_loader, device, mm_per_pixel)
        val_mre = (evaluate(probe, val_loader, device, mm_per_pixel)["MRE"]
                   if use_val else metrics["MRE"])
        sel = val_mre
        record = {"epoch": epoch, "train_loss": ep_loss / max(nb, 1),
                  "val_MRE": val_mre, **metrics}
        history.append(record)
        if epoch % args.log_every == 0 or epoch == args.epochs - 1:
            unit = "mm"
            print(f"epoch={epoch:3d} loss={record['train_loss']:.4f} "
                  f"val_MRE={val_mre:.3f}{unit} "
                  f"test_MRE={metrics['MRE']:.3f}{unit} "
                  f"SDR@2mm={metrics['SDR@2.0mm']:.1f}% "
                  f"P95={metrics['P95']:.3f}{unit}")

        # Early stopping on the SELECTION metric (val MRE); keep best-so-far head.
        if sel < best_sel - 1e-6:
            best_sel = sel
            best_state = copy.deepcopy(
                {k: v.detach().cpu() for k, v in probe.state_dict().items()}
            )
            bad_epochs = 0
        else:
            bad_epochs += 1
            if patience and bad_epochs >= patience:
                print(f"early stop at epoch {epoch} (best {sel_name} "
                      f"MRE={best_sel:.3f}, no improvement for {patience} epochs)")
                break

    if best_state is not None:
        probe.load_state_dict(best_state)
        print(f"restored best epoch by {sel_name} (MRE={best_sel:.3f})")

    if args.out:
        torch.save(probe.head.state_dict(), args.out)
    return probe, history

@torch.no_grad()
def visualize_predictions(probe, test_ds, image_size, n=6, out=None, title=None,
                          seed=0):
    """Overlay predicted (red x) vs ground-truth (green o) landmarks on a few
    held-out test images for a trained probe. Yellow lines show per-point error.
    Returns the matplotlib Figure (also saved to ``out`` if given)."""
    import matplotlib.pyplot as plt

    device = next(probe.parameters()).device
    probe.eval()
    n = min(n, len(test_ds))
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(test_ds), generator=g).tolist()[:n]

    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows),
                             squeeze=False)
    for a in axes.ravel():
        a.axis("off")

    for a, i in zip(axes.ravel(), idx):
        x, gt = test_ds[i]
        xb = x.unsqueeze(0).to(device)
        pred = heatmaps_to_coords(probe(xb))[0].cpu()   # (K, 2) as (x, y)
        gt = gt.float()
        a.imshow(x[0].cpu(), cmap="gray")
        for (gx, gy), (px, py) in zip(gt, pred):
            a.plot([gx, px], [gy, py], c="yellow", lw=0.6, alpha=0.8)
        a.scatter(gt[:, 0], gt[:, 1], s=20, c="lime", marker="o",
                  edgecolors="black", linewidths=0.4)
        a.scatter(pred[:, 0], pred[:, 1], s=24, c="red", marker="x")
        px_mre = torch.linalg.norm(pred - gt, dim=-1).mean().item()
        a.set_title(f"test idx {i}  px-MRE={px_mre:.1f}", fontsize=9)

    handles = [
        plt.Line2D([], [], marker="o", color="lime", ls="", label="ground truth"),
        plt.Line2D([], [], marker="x", color="red", ls="", label="predicted"),
    ]
    fig.legend(handles=handles, loc="upper right")
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    if out:
        fig.savefig(out, dpi=120)
    return fig


In [ ]:
import importlib
from types import SimpleNamespace

import cfm_pretrain
import probe_landmark_detection

# Pick up the freshly written modules (the %%writefile cells above).
importlib.reload(cfm_pretrain)
importlib.reload(probe_landmark_detection)
from probe_landmark_detection import (
    build_eval_split,
    run_probe,
    visualize_predictions,
)


def best(history):
    """Best epoch record, selected on VAL MRE (falls back to test MRE)."""
    return min(history, key=lambda r: r.get("val_MRE", r["MRE"]))


def make_probe_args(**overrides):
    """Build a downstream-probe arg namespace from the shared CONTROLLED
    variables in EXPERIMENT, overriding only the dataset-specific fields.

    Every probe (real-landmark, ISBI2015, frozen) reads the same controls from
    here so the backbone init stays the ONLY difference across compared runs.
    Pass dataset-specific keys (dataset/root/landmarks_dir/...) as ``overrides``.
    """
    base = dict(
        split="train",
        num_classes=E.num_classes,
        freeze_backbone=E.freeze_backbone,
        image_size=E.image_size,
        batch_size=E.batch_size,
        epochs=E.probe_epochs,
        n_shot=E.n_shot,
        test_frac=E.test_frac,
        sigma=E.sigma,
        lr=E.probe_lr,
        # Fixed forward-t for feature extraction. CFM probes the clean
        # data endpoint (t=1.0). The DDPM backbone follows CDPM's t=200 of
        # T=500, which on this repo's continuous axis (t=1 clean, t=0
        # noise) is t ~ 1 - 200/499 ~ 0.6. Overridable via EXPERIMENT['probe_t'].
        t_value=getattr(E, "probe_t", None)
                or (0.6 if getattr(E, "backbone", "cfm") == "ddpm" else 1.0),
        mm_per_pixel=1.0,
        log_every=10,
        out=None,
        seed=E.seed,
        loss=E.probe_loss,
        weight_decay=E.probe_weight_decay,
        patience=E.probe_patience,
    )
    base.update(overrides)
    return SimpleNamespace(**base)


## 6. Real anatomical landmarks (solving the pseudo-landmark caveat)

The probe above used auto pseudo-landmarks (a plumbing check). Here we upgrade to **real lung landmarks** from the [ngaggion Chest-xray-landmark-dataset](https://github.com/ngaggion/Chest-xray-landmark-dataset) (RL 44 + LL 50 = 94 points), which cover the Shenzhen set.

The annotations live in a **1024×1024 frame** produced by *crop-to-content → pad-to-square → resize*. So we replicate that exact crop+pad on each raw Shenzhen image; after our dataset resizes to `image_size`, a landmark maps by the simple scale `image_size / 1024` (passed as `landmark_ref_size=1024`). The next cells write the converter, run it, sanity-check the alignment, and re-run the probe (pretrained vs baseline) on real anatomy.

> Shenzhen ships **no pixel spacing**, so errors below are reported in *pixels* — but the task is now genuine anatomical localization, so the pretrained-vs-baseline gap is meaningful. For millimetre MRE/SDR, use the ISBI2015 cephalometric benchmark (known 0.1 mm/px) with the same probe.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Sanity overlay: landmarks (scaled 1024 -> display size) on a preprocessed image.
stem = sorted(Path(LM_DIR).glob("*.npy"))[0].stem
img = cv2.imread(f"{PP_ROOT}/{stem}.png", cv2.IMREAD_GRAYSCALE)
lm = np.load(f"{LM_DIR}/{stem}.npy")  # (94, 2) in 1024 space
scale = img.shape[0] / 1024.0

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="gray")
plt.scatter(lm[:, 0] * scale, lm[:, 1] * scale, s=6, c="lime")
plt.title(f"{stem}: 94 lung landmarks (should hug both lungs)")
plt.axis("off")
plt.show()


In [ ]:
# Re-run the probe on REAL lung landmarks: pretrained vs random-init baseline.
# Uses the crop+pad-aligned images (PP_ROOT) + combined 94-pt landmarks (LM_DIR).
# Shenzhen ships no pixel spacing, so mm_per_pixel=1.0 => errors are in pixels.

def make_real_probe_args(backbone_ckpt):
    return make_probe_args(
        dataset="chest",            # real ngaggion lung landmarks
        root=PP_ROOT,               # crop+pad-aligned images
        landmarks_dir=LM_DIR,       # combined RL+LL (94, 2) per image
        landmark_ref_size=1024,     # annotations are in the 1024 frame
        backbone_ckpt=backbone_ckpt,  # None -> from-scratch baseline
    )


print("=== Pretrained backbone (real landmarks) ===")
probe_pre_real, hist_pre_real = run_probe(make_real_probe_args(args.out))
print("\n=== Random-init baseline (real landmarks) ===")
_, hist_rnd_real = run_probe(make_real_probe_args(None))

best_pre, best_rnd = best(hist_pre_real), best(hist_rnd_real)
log_result("chest-real/pretrained", best_pre,
           extra={"probe": "chest-real", "protocol": "finetune",
                  "init": "pretrained", "unit": "px"})
log_result("chest-real/random", best_rnd,
           extra={"probe": "chest-real", "protocol": "finetune",
                  "init": "random", "unit": "px"})

print("\n" + "=" * 52)
print(f"{'metric':<12}{'pretrained':>14}{'random-init':>16}")
print("-" * 52)
for key in ("MRE", "P95", "SDR@2.0mm", "SDR@4.0mm"):
    print(f"{key:<12}{best_pre[key]:>14.3f}{best_rnd[key]:>16.3f}")
print("=" * 52)
print("(real lung landmarks; MRE/P95 in pixels @128 -> multiply by 1024/128 "
      "for 1024-frame px)")

# Metric curves on real anatomy.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for hist, name in [(hist_pre_real, "pretrained"), (hist_rnd_real, "random-init")]:
    epochs = [h["epoch"] for h in hist]
    ax[0].plot(epochs, [h["MRE"] for h in hist], label=name)
    ax[1].plot(epochs, [h["SDR@2.0mm"] for h in hist], label=name)
ax[0].set(xlabel="epoch", ylabel="MRE (px)",
          title="Real-landmark test MRE (lower=better)")
ax[1].set(xlabel="epoch", ylabel="SDR@2px (%)",
          title="Real-landmark test SDR@2 (higher=better)")
for a in ax:
    a.legend()
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "probe_metrics_real.png"), dpi=120)
plt.show()


## 6.5 True-millimetre metrics on ISBI2015 (head-to-head vs CDPM-Align)

§6 reports MRE in **pixels** because Shenzhen ships no pixel spacing — so its number is *not* comparable to CDPM's **1.54 mm**. This section runs the *same* probe on the **ISBI2015 cephalometric** benchmark (19 landmarks) using a loader that is **faithful to CDPM-Align** (`datasets/cephalo_dataset.py`), so the result is a true millimetre head-to-head.

**What the `ISBI2015LandmarkDataset` matches from CDPM:**
- **Split by sorted filename** (001–400): `train=[:130]`, `val=[130:150]`, `test=[150:400]` (250 eval images). The few-shot pool is drawn (seeded) from the 130 train images; evaluation is always the fixed 250-image test set.
- **Per-axis millimetres.** Error is measured in `image_size`-px space then converted **per axis**: `mm_per_pixel = [(W_native/S)×0.1, (H_native/S)×0.1]`. ISBI images are non-square (~1935×2400), so x and y mm/px differ ~20% — `compute_metrics` multiplies the (pred−gt) displacement by this length-2 vector **before** the norm. A single scalar would be wrong here.
- **19 landmarks**, native pixel spacing **0.1 mm**.

The probe cell auto-discovers the image dir (`ISBI_ROOT`) and the landmark CSV dir (`ISBI_CSV_DIR`) from the attached Kaggle input and no-ops with a message if the dataset isn't present.

> **Annotations:** the public Kaggle mirror [`jiahongqian/cephalometric-landmarks`](https://www.kaggle.com/datasets/jiahongqian/cephalometric-landmarks) ships **senior-reader** annotations only (`*_senior.csv`). CDPM averages junior+senior — using senior-only is a small, documented deviation. `image_size` is now **256** (matching CDPM), so mm/px ≈ `[0.756, 0.938]`.

In [ ]:
# True-mm DOWNSTREAM probe on ISBI2015 -- the head-to-head vs CDPM-Align.
# Faithful to CDPM's cephalo protocol: sorted-filename split (few-shot pool
# train=[:130], fixed eval test=[150:400] = 250 imgs), 19 landmarks, per-axis
# mm from native 0.1 mm/px. Landmarks come from the attached
# jiahongqian/cephalometric-landmarks *_senior.csv files (senior reader only;
# CDPM averages junior+senior -- a small documented deviation).
if not ISBI_ROOT or not ISBI_CSV_DIR:
    print("[skip] ISBI2015 images/landmarks not found. Attach "
          "jiahongqian/cephalometric-landmarks to run the mm head-to-head.")
else:
    def make_isbi_probe_args(backbone_ckpt, freeze=None):
        overrides = dict(
            dataset="isbi2015",
            root=ISBI_ROOT,
            landmarks_dir=ISBI_CSV_DIR,
            landmark_ref_size=2400,
            phase="train",
            backbone_ckpt=backbone_ckpt,
        )
        if freeze is not None:
            overrides["freeze_backbone"] = freeze
        return make_probe_args(**overrides)

    print("=== Pretrained backbone (ISBI2015, mm) ===")
    probe_pre_isbi, hist_pre_isbi = run_probe(make_isbi_probe_args(args.out))
    print("\n=== Random-init baseline (ISBI2015, mm) ===")
    _, hist_rnd_isbi = run_probe(make_isbi_probe_args(None))

    best_pre, best_rnd = best(hist_pre_isbi), best(hist_rnd_isbi)
    log_result("isbi2015/pretrained", best_pre,
               extra={"probe": "isbi2015", "protocol": "finetune",
                      "init": "pretrained", "unit": "mm"})
    log_result("isbi2015/random", best_rnd,
               extra={"probe": "isbi2015", "protocol": "finetune",
                      "init": "random", "unit": "mm"})

    print("\n" + "=" * 52)
    print(f"{'metric':<12}{'pretrained':>14}{'random-init':>16}")
    print("-" * 52)
    for key in ("MRE", "P95", "SDR@2.0mm", "SDR@4.0mm"):
        print(f"{key:<12}{best_pre[key]:>14.3f}{best_rnd[key]:>16.3f}")
    print("=" * 52)
    print("(ISBI2015 cephalometric; MRE/P95 in mm; %d-shot. CDPM-Align 25-shot "
          "ref: 1.54 mm MRE, 77.52%% SDR@2mm)" % E.n_shot)


## 6.6 Qualitative check: landmark predictions from the fine-tuned model

The tables above report the *numbers* (MRE / SDR). This section shows the
**actual predictions** of the fine-tuned probe on its held-out test images:
green circles = ground-truth landmarks, red crosses = model predictions, and
yellow lines link each prediction to its target (shorter = better). We reuse the
deterministic `build_eval_split` so these are exactly the images the probe was
scored on. Prefers the ISBI2015 probe (true-mm head-to-head) when available,
otherwise falls back to the real lung-landmark probe.

In [ ]:
# Visualise predicted vs ground-truth landmarks for the FINE-TUNED probe.
# (Reuses the seeded test split; green=GT, red=pred, yellow=error segment.)
if "probe_pre_isbi" in globals():
    viz_probe = probe_pre_isbi
    viz_args = make_isbi_probe_args(args.out)
    viz_title = "Fine-tuned CFM probe -- ISBI2015 test (green=GT, red=pred)"
    viz_tag = "isbi"
elif "probe_pre_real" in globals():
    viz_probe = probe_pre_real
    viz_args = make_real_probe_args(args.out)
    viz_title = "Fine-tuned CFM probe -- real lung landmarks test (green=GT, red=pred)"
    viz_tag = "real"
else:
    viz_probe = None

if viz_probe is None:
    print("[skip] no fine-tuned probe in scope -- run the downstream probe cells first.")
else:
    _, _, viz_test_ds, _, _ = build_eval_split(viz_args)
    fig = visualize_predictions(
        viz_probe, viz_test_ds, E.image_size, n=6,
        out=os.path.join(OUT_DIR, f"landmark_predictions_{viz_tag}.png"),
        title=viz_title,
    )
    plt.show()
    print(f"Saved landmark_predictions_{viz_tag}.png ({len(viz_test_ds)} test images)")


## 6.7 Frozen linear-probe: isolating representation quality

The §6.5 head-to-head uses CDPM's **full fine-tune**, which lets *both* the
pretrained and random-init backbones fit the 25 training shots to a similar
minimum — so it barely separates them (4.86 vs 4.92 mm). To ask the cleaner
question *"are the CFM features themselves better than random features?"* we
**freeze the backbone** and train only the multi-scale head. Here the backbone
init is the *only* difference, so any gap is pure representation quality.

Verdict logic: if frozen-pretrained clearly beats frozen-random, CFM pretraining
*does* learn landmark-relevant structure and the full-fine-tune washout is the
reason the head-to-head looks flat; if they tie, the pretraining objective isn't
yet producing useful features (a deeper issue to chase).

In [ ]:
# Frozen linear-probe on ISBI2015 (mm): backbone NOT updated, head only.
# Isolates representation quality (backbone init is the only difference).
if "make_isbi_probe_args" in globals():
    print("=== FROZEN probe: pretrained backbone (ISBI2015, mm) ===")
    _, hist_frozen_pre = run_probe(make_isbi_probe_args(args.out, freeze=True))
    print("\n=== FROZEN probe: random-init backbone (ISBI2015, mm) ===")
    _, hist_frozen_rnd = run_probe(make_isbi_probe_args(None, freeze=True))

    frozen_pre, frozen_rnd = best(hist_frozen_pre), best(hist_frozen_rnd)
    log_result("isbi2015-frozen/pretrained", frozen_pre,
               extra={"probe": "isbi2015-frozen", "protocol": "linear-probe",
                      "init": "pretrained", "unit": "mm"})
    log_result("isbi2015-frozen/random", frozen_rnd,
               extra={"probe": "isbi2015-frozen", "protocol": "linear-probe",
                      "init": "random", "unit": "mm"})

    # 2x2 summary: {frozen, fine-tune} x {pretrained, random}. Fine-tune numbers
    # come from the §6.5 run (best() of those histories, if present).
    print("\n" + "=" * 68)
    print(f"{'ISBI2015 25-shot (mm)':<24}{'pretrained':>14}{'random':>12}{'gain (mm)':>16}")
    print("-" * 68)
    print(f"{'frozen  MRE':<24}{frozen_pre['MRE']:>14.3f}{frozen_rnd['MRE']:>12.3f}"
          f"{frozen_rnd['MRE'] - frozen_pre['MRE']:>16.3f}")
    print(f"{'frozen  SDR@2mm':<24}{frozen_pre['SDR@2.0mm']:>14.2f}"
          f"{frozen_rnd['SDR@2.0mm']:>12.2f}"
          f"{frozen_pre['SDR@2.0mm'] - frozen_rnd['SDR@2.0mm']:>16.2f}")
    if "hist_pre_isbi" in globals() and "hist_rnd_isbi" in globals():
        ft_pre, ft_rnd = best(hist_pre_isbi), best(hist_rnd_isbi)
        print(f"{'finetune  MRE':<24}{ft_pre['MRE']:>14.3f}{ft_rnd['MRE']:>12.3f}"
              f"{ft_rnd['MRE'] - ft_pre['MRE']:>16.3f}")
        print(f"{'finetune  SDR@2mm':<24}{ft_pre['SDR@2.0mm']:>14.2f}"
              f"{ft_rnd['SDR@2.0mm']:>12.2f}"
              f"{ft_pre['SDR@2.0mm'] - ft_rnd['SDR@2.0mm']:>16.2f}")
    print("=" * 68)
    print("gain>0 (MRE) or gain>0 (SDR) => pretraining helps under that protocol.")

    # MRE curves: frozen pretrained vs frozen random.
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for hist, name in [(hist_frozen_pre, "frozen pretrained"),
                       (hist_frozen_rnd, "frozen random")]:
        epochs = [h["epoch"] for h in hist]
        ax[0].plot(epochs, [h["MRE"] for h in hist], label=name)
        ax[1].plot(epochs, [h["SDR@2.0mm"] for h in hist], label=name)
    ax[0].set(xlabel="epoch", ylabel="MRE (mm)",
              title="Frozen-probe test MRE (lower=better)")
    ax[1].set(xlabel="epoch", ylabel="SDR@2mm (%)",
              title="Frozen-probe test SDR@2mm (higher=better)")
    for a in ax:
        a.legend()
        a.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "probe_metrics_isbi_frozen.png"), dpi=120)
    plt.show()
else:
    print("[skip] ISBI2015 not attached -- frozen representation probe needs the "
          "jiahongqian/cephalometric-landmarks dataset.")


## 7. Sampling check: generate radiographs

A generative pretrainer should be able to *synthesise* plausible images, not just learn transferable features. This section integrates the learned ODE `dx/dt = v(x, t)` from Gaussian noise (`t=0`) to data (`t=1`) with an explicit Euler solver.

The backbone was pretrained with **dataset-index class conditioning** (CDPM-Align style), so we sample two ways for comparison:

- **Unconditional** (top row): the null/uncond token (class 0) — the *hardest* case, and what the earlier smoky/anatomy-free grids used.
- **Chest-conditional with classifier-free guidance** (bottom row): `v = v_uncond + w·(v_cond − v_uncond)` with the Shenzhen-chest class and guidance `w`. This steers the trajectory toward the chest manifold and yields clearer anatomy — the same conditional-generation mechanism CDPM relies on.

> **Caveat (scientific honesty):** conditioning + CFG makes samples *look* more chest-like, but the dominant reason earlier samples were structureless is **backbone capacity + pretraining compute**, not the sampler. The definitive fix is re-pretraining the scaled-up backbone (base 64, 3 levels, bottleneck attention) for more iterations; expect real ribs/lung fields only after that. Sampling is auto-robust to `base_channels`/`num_classes` (read from the checkpoint), so it works on both old and new backbones.


In [ ]:
import torch
import matplotlib.pyplot as plt
from deltaflow.models.conditional_unet import ConditionalUNetVelocityField

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- sampling knobs (all default to the ORIGINAL behavior; opt in to test) ---
N_STEPS = 200          # ODE steps. Raise to 400 to test discretization error.
SAMPLER = "euler"      # "euler" (1st-order, default) or "heun" (2nd-order).
USE_EMA = False        # load EMA shadow weights (*.ema.pt) if present.


def resolve_checkpoint_path(out_path, use_ema):
    """Return the weights file to sample from: the EMA sibling (*.ema.pt) when
    ``use_ema`` and it exists, else the primary ``out_path``."""
    if not use_ema:
        return out_path
    root, ext = os.path.splitext(out_path)
    ema_path = root + ".ema" + (ext or ".pt")
    if os.path.exists(ema_path):
        print("using EMA weights:", ema_path)
        return ema_path
    print("USE_EMA set but no EMA file found; using", out_path)
    return out_path


# Load the pretrained velocity field. Auto-detect BOTH the class table size
# (class_embed.weight) AND base_channels (stem.weight out-channels) from the
# checkpoint, so this cell works unchanged on the old 32-ch backbone and the new
# scaled-up 64-ch / 3-level / attention backbone.
ckpt_path = resolve_checkpoint_path(args.out, USE_EMA)
state = torch.load(ckpt_path, map_location=device)
num_classes_ckpt = (state["class_embed.weight"].shape[0]
                    if "class_embed.weight" in state
                    else getattr(E, "num_classes", 0))
base_channels_ckpt = state["stem.weight"].shape[0]

gen = ConditionalUNetVelocityField(
    in_channels=1, cond_channels=1,
    base_channels=base_channels_ckpt, num_classes=num_classes_ckpt,
).to(device)
gen.load_state_dict(state)
gen.eval()
print(f"loaded backbone: base_channels={base_channels_ckpt}  "
      f"num_classes={num_classes_ckpt}")


@torch.no_grad()
def cfm_sample(model, n, size, n_steps=200, class_idx=None, guidance=0.0,
               seed=0, sampler="euler"):
    """Explicit-Euler integration of dx/dt = v(x, t) from noise (t=0) to data
    (t=1). With ``class_idx`` and ``guidance > 0`` applies classifier-free
    guidance: v = v_uncond + guidance * (v_cond - v_uncond). ``class_idx=None``
    (or guidance=0) samples unconditionally via the null token 0."""
    generator = torch.Generator(device=device).manual_seed(seed)
    x = torch.randn(n, 1, size, size, generator=generator, device=device)
    dt = 1.0 / n_steps
    null = torch.zeros(n, dtype=torch.long, device=device)
    cls = None if class_idx is None else torch.full(
        (n,), class_idx, dtype=torch.long, device=device)

    def velocity(state_x, t):
        if cls is not None and guidance > 0:
            v_cond = model(state_x, t, class_idx=cls)
            v_uncond = model(state_x, t, class_idx=null)
            return v_uncond + guidance * (v_cond - v_uncond)
        return model(state_x, t, class_idx=cls)  # cls None -> null token

    for i in range(n_steps):
        t0 = torch.full((n,), i * dt, device=device)
        v0 = velocity(x, t0)
        if sampler == "heun" and i < n_steps - 1:
            # 2nd-order predictor-corrector: average the slope at both ends.
            t1 = torch.full((n,), (i + 1) * dt, device=device)
            v1 = velocity(x + v0 * dt, t1)
            x = x + 0.5 * (v0 + v1) * dt
        else:
            x = x + v0 * dt
    return x.clamp(0, 1).cpu()


N = 4
GUIDANCE = 2.0
CHEST_CLASS_IDX = E.dataset_index["shenzhen"]  # dataset index 1
# Guard if the backbone is unconditional (no chest class in the table).
chest_cls = CHEST_CLASS_IDX if num_classes_ckpt > CHEST_CLASS_IDX else None

uncond = cfm_sample(gen, N, args.image_size, n_steps=N_STEPS,
                    class_idx=None, guidance=0.0, sampler=SAMPLER)
cond = cfm_sample(gen, N, args.image_size, n_steps=N_STEPS,
                  class_idx=chest_cls, guidance=GUIDANCE, sampler=SAMPLER)

fig, axes = plt.subplots(2, N, figsize=(3 * N, 6))
for ax, img in zip(axes[0], uncond):
    ax.imshow(img[0], cmap="gray")
for ax, img in zip(axes[1], cond):
    ax.imshow(img[0], cmap="gray")
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_ylabel("unconditional\n(null token)", fontsize=10)
axes[1, 0].set_ylabel(f"chest-conditional\n(CFG w={GUIDANCE})"
                      if chest_cls is not None else "unconditional", fontsize=10)
fig.suptitle(f"CFM samples ({SAMPLER} ODE, {N_STEPS} steps"
             f"{' +EMA' if USE_EMA else ''}): "
             "top=unconditional | bottom=chest+CFG")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "samples.png"), dpi=120)
plt.show()


## 8. Next steps

You now have the full **pretrain → probe → real-landmark eval → sampling** loop: a CFM velocity field pretrained with multi-scale guidance (delta) alignment, a few-shot landmark-detection probe with a pretrained-vs-baseline comparison on pseudo-, real-lung, and **ISBI2015 cephalometric (millimetre)** landmarks, and an unconditional generation check. To turn this PoC into DPhil-application evidence:

1. **Head-to-head vs CDPM-Align (implemented in §6.5).** Same data + budget, **DDPM-vs-CFM backbone** under the *same* probe, on ISBI2015 in millimetres. Run `backbone="ddpm"` vs `"cfm"` — the single independent variable. CDPM-Align ISBI2015 25-shot reference: **1.54 mm MRE, 77.52% SDR@2mm**.
2. **Ablations.** `lambda_align ∈ {0, 1, 5}` at pretraining (isolate the alignment term), linear-probe vs `freeze_backbone=False` fine-tune, and `n_shot ∈ {10, 25}`.
3. **Sharper samples.** Scale `n_shot`/`epochs` and swap the Euler `FlowSampler` for `HeunSolver` (§7) for higher-fidelity generated radiographs, and try *conditional* sampling by passing a landmark/dataset-index condition.